# PRDM9 Hotspots

Aleva et al. (GSE166483) hotspots (hg38) → liftover → hs1, then paired GC-matched controls, eG4 coverage.

In [ ]:
import os
import bisect
import random
import subprocess
import tempfile
import shutil
from io import StringIO
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pybedtools import BedTool
from scipy.stats import fisher_exact, wilcoxon
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
import gzip
from Bio.SeqIO.FastaIO import SimpleFastaParser

def parse_fasta(fasta):
    with gzip.open(fasta, "rt") as f:
        for name, seq in SimpleFastaParser(f):
            yield name.split()[0], seq.lower()

In [ ]:
SCRATCH       = Path(os.getenv("SCRATCH"))
WORK          = Path(os.getenv("WORK"))
DATA_DIR      = SCRATCH / "g4_t2t_revisions_data"
CTRL_OUT      = DATA_DIR / "PRMD9_hotspot_controls"
LIFTED_DIR    = DATA_DIR / "PRMD9_hotspots_lifted"

HOTSPOT_FILE  = WORK.joinpath("g4_t2t_revisions/PRMD9_hotspots_Aleva_et_al/GSE166483_hotspotsData.txt.gz")
FASTA_HS1     = DATA_DIR / "fasta" / "hs1.fa.gz"
CHAIN_PATH    = WORK.joinpath("g4_t2t_revisions/hg38ToHs1.over.chain.gz")
LIFTOVER_BIN  = Path("/home1/10904/nikolchanchan/micromamba/envs/g4_t2t/bin/liftOver")

CLEAN = True
for d in [CTRL_OUT, LIFTED_DIR]:
    # shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)

assert HOTSPOT_FILE.exists(), HOTSPOT_FILE
assert FASTA_HS1.exists(),    FASTA_HS1
assert CHAIN_PATH.exists(),   CHAIN_PATH
assert LIFTOVER_BIN.exists(), LIFTOVER_BIN
print("All paths OK")

In [ ]:
import os
indir = Path(f"{os.getenv('SCRATCH')}/g4_t2t_revisions_data")
g4_hg19_df = pd.read_table(f"{indir}/EndoQuad/Predicted_human_G4.txt")
FIELDS = ["Chr", "Start", "End"]
eg4_df = g4_hg19_df[g4_hg19_df["Confidence level"] != "Non-eG4"].copy()[FIELDS].rename(columns={"Chr": "seqID", "Start": "start", "End": "end"})
eg4_df

In [ ]:
eg4_df.to_csv(DATA_DIR / "eG4_HG19.txt", 
            sep="\t", 
            index=False, 
            header=False)

In [ ]:
from subprocess import run

eg4_hs1 = LIFTED_DIR / 'eG4_HS1.txt'
if eg4_hs1.is_file():
    print("eG4_HS1.txt already exists")
    eg4_hs1_df = pd.read_table(eg4_hs1, header=None, names=["seqID", "start", "end"])
else:
    CHAIN_HG19_TO_HS1 = Path("./hg19ToHs1.over.chain.gz")
    run(f"liftOver {DATA_DIR / 'eG4_HG19.txt'} {CHAIN_HG19_TO_HS1} {LIFTED_DIR / 'eG4_HS1.txt'} {LIFTED_DIR / 'eG4_HS1_unmapped.txt'}", shell=True, check=True)

In [ ]:
from subprocess import run
LINK = "https://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz"
CHAIN_HG19_TO_HG38 = Path("hg19ToHg38.over.chain.gz")
if not CHAIN_HG19_TO_HG38.is_file():
    os.system(f"wget {LINK}")

eg4_hg38 = LIFTED_DIR / 'eG4_HG38.txt'
if eg4_hg38.is_file():
    print("eG4_HG38.txt already exists")
    eg4_hg38_df = pd.read_table(eg4_hg38, header=None, names=["seqID", "start", "end"])
else:
    command = f"liftOver {DATA_DIR / 'eG4_HG19.txt'} {CHAIN_HG19_TO_HG38} {LIFTED_DIR / 'eG4_HG38.txt'} {LIFTED_DIR / 'eG4_HG38_unmapped.txt'}"
    print(command)
    # run(f"liftOver {DATA_DIR / 'eG4_HG19.txt'} {CHAIN_HG19_TO_HG38} {LIFTED_DIR / 'eG4_HG38.txt'} {LIFTED_DIR / 'eG4_HG38_unmapped.txt'}", shell=True, check=True)

In [ ]:
# 6f2a8fc4bed6a223e1bd34a863cff466  GSE166483_hotspotsData.tab md5sum

In [ ]:
target = Path(os.getenv("WORK")).joinpath("figures_g4_t2t_NEW/PRMD9_data")
target_fig = target.joinpath("figure")
target_data = target.joinpath("data")
target_fig.mkdir(parents=True, exist_ok=True)
target_data.mkdir(parents=True, exist_ok=True)

## 1. Load hotspot data

In [ ]:
hotspot_df = pd.read_table(HOTSPOT_FILE, comment="#")
hotspot_df = hotspot_df.rename(columns={"cs": "chrom", 
                                        "from": "start", 
                                        "to": "end"})
print(hotspot_df.shape)
hotspot_df.head()

In [ ]:
ALLELE_COLS = ["AA1", "AA2", "AA3", "AA4", "AB1", "AC", "AN", "CL4"]
print("Hotspots active per allele (strength > 0):")
for col in ALLELE_COLS:
    n = (hotspot_df[col] > 0).sum()
    print(f"  {col}: {n:,}")

## 2. Hotspot strength distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.patch.set_facecolor("white")

for ax, allele in zip(axes.flat, ALLELE_COLS):
    vals = hotspot_df.loc[hotspot_df[allele] > 0, allele]
    ax.hist(vals, bins=60, color="#2166ac", edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Hotspot strength", fontsize=13)
    ax.set_ylabel("Count", fontsize=13)
    ax.set_title(allele, fontsize=14, fontweight="bold")
    ax.tick_params(labelsize=11)
    ax.grid(lw=0.4, alpha=0.6)
    ax.set_yscale("log", base=10)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    med = vals.median()
    ax.axvline(med, ls="--", lw=1.5, color="red", label=f"median={med:.0f}")
    ax.legend(fontsize=10, frameon=False)

plt.tight_layout()
fig.savefig(target_fig / "prmd9_hotspot_strength_distributions.pdf",
            bbox_inches="tight", 
            transparent=True)
plt.show()

In [ ]:
print("Hotspot size distribution (bp):")
hotspot_df["size"] = hotspot_df["end"] - hotspot_df["start"]
print(hotspot_df["size"].describe())

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(hotspot_df["size"], bins=60, color="#2166ac", edgecolor="black", linewidth=0.4)
ax.set_xlabel("Hotspot size (bp)", fontsize=14)
ax.set_ylabel("Count", fontsize=14)
ax.grid(lw=0.4, alpha=0.6)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_yscale("log", base=10)
plt.tight_layout()
fig.savefig(target_fig / "prmd9_hotspot_size_distribution.pdf",
            bbox_inches="tight", transparent=True)
plt.show()

## 3. Split by allele and liftover hg38 → hs1

In [ ]:
import tempfile
import shutil

def liftover_df(df_hg38, chain, col_names=("seqID", "start", "end", "score")):
    """Liftover a BED DataFrame (chrom, start, end, +extra) and return hs1 DataFrame."""
    tmpdir = Path(tempfile.mkdtemp())
    tmp_in, tmp_out = tmpdir / "in.bed", tmpdir / "out.bed"
    unmap = tmpdir / "unmapped.bed"
    df_hg38.to_csv(tmp_in, sep="\t", index=False, header=False)

    proc = subprocess.run(
        ["liftOver", str(tmp_in), str(chain), str(tmp_out), str(unmap)],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        shutil.rmtree(tmpdir)
        raise RuntimeError(proc.stderr.strip())

    names = list(col_names)[: df_hg38.shape[1]]
    lifted = pd.read_table(tmp_out, header=None, names=names)

    n_unmapped = 0
    if unmap.exists():
        with open(unmap) as fh:
            n_unmapped = sum(1 for line in fh if not line.startswith("#"))

    shutil.rmtree(tmpdir)
    return lifted, n_unmapped


def load_or_liftover_hotspots(hotspot_df, allele_cols, chain, lifted_dir, force=False):
    result = {}
    for allele in allele_cols:
        cache = lifted_dir / f"hotspot_{allele}_hs1.tsv.gz"
        if cache.exists() and not force:
            result[allele] = pd.read_table(cache)
            print(f"  {allele}: loaded from cache ({len(result[allele]):,} hotspots)")
            continue
        sub = hotspot_df.loc[hotspot_df[allele] > 0, ["chrom", "start", "end", allele]].copy()
        print(f"  {allele}: lifting {len(sub):,} hotspots...")
        lifted, n_unmapped = liftover_df(sub, chain)
        print(f"    → {len(lifted):,} mapped, {n_unmapped:,} unmapped")
        assert len(lifted) + n_unmapped == len(sub), \
        f"{allele}: {len(lifted)} + {n_unmapped} != {len(sub)}"
        lifted.to_csv(cache, 
                      sep="\t", 
                      index=False, 
                      header=True,
                      compression="gzip")
        result[allele] = lifted
    return result

FORCE = True
hotspots_hs1 = load_or_liftover_hotspots(
    hotspot_df, 
    ALLELE_COLS, 
    CHAIN_PATH, 
    target_data, 
    force=FORCE
)
hotspots_hs1["AA1"].head()

In [ ]:
alleles = list(hotspot_df.columns[3:])
alleles

In [ ]:
hotspot_df["length"] = hotspot_df["end"] - hotspot_df["start"]
hotspot_df["length"].max()

In [ ]:
genome_length = 0
FASTA_HG38 = WORK.joinpath("g4_revisions/g4_t2t_revisions/hg38.fa.gz")
for seqID, seq in parse_fasta(FASTA_HG38):
    seq = seq.lower()
    genome_length += sum(seq.count(base) for base in "acgt")


In [ ]:
gw_densities = {}
datasets = [("eG4", eg4_hg38_df)]
datasets_bed = [(name, BedTool.from_dataframe(df)) for name, df in datasets]
for name, bed in datasets_bed:
    total_length = pd.read_table(bed.merge().fn, header=None, names=["seqID", "start", "end"]).eval("length = end - start")["length"].sum()
    print(total_length)
    density = 1e3 * total_length / genome_length
    gw_densities[name] = density
    print(f"{name}: {total_length:,} bp in {len(bed):,} intervals, "
          f"genome length {genome_length:,} bp, density {density:.6f}")

In [ ]:
hotspots_hg38 = []
for allele in alleles:
    hotspot_temp = pd.read_table(
        BedTool.from_dataframe(hotspot_df.loc[hotspot_df[allele] > 0, ["chrom", "start", "end"]].copy()).sort().merge().fn,
        header=None,
        names=["seqID", "start", "end"]
    )
    hotspot_temp.loc[:, "allele"] = allele
    hotspots_hg38.append(hotspot_temp)
hotspots_hg38_df = pd.concat(hotspots_hg38, ignore_index=True)
hotspots_hg38_bed = BedTool.from_dataframe(hotspots_hg38_df).sort()

coverage_df = []
COVERAGE_FIELDS = ["total_hits", "total_bp", "compartment_length", "coverage"]
for name, bed in datasets_bed:
    coverage_part_df = (
        pd.read_table(
            hotspots_hg38_bed.coverage(bed).fn,
            header=None,
            names=["seqID", "start", "end", "allele"] + COVERAGE_FIELDS
        )
        .assign(at_least_one=lambda ds: (ds["total_hits"] > 0).astype(int))
        .groupby("allele")
        .agg(
            {
            "total_bp": "sum",
             "compartment_length": "sum",
             "total_hits": "sum",
             "at_least_one": lambda ds: 1e2 * ds.mean()
             }
        )
        .assign(density=lambda x: x["total_bp"] * 1e3 / x["compartment_length"])
    )
    coverage_part_df.loc[:, "fold_enrichment"] = coverage_part_df["density"] / gw_densities[name]
    coverage_part_df.loc[:, "dataset"] = name
    coverage_df.append(coverage_part_df)
coverage_df = pd.concat(coverage_df, ignore_index=True)
coverage_df

In [ ]:
import subprocess
import urllib.request
import gzip
import shutil
import tempfile
import os
from pathlib import Path

CHAIN_PATH  = Path(os.environ["WORK"]) / "hg38ToHs1.over.chain.gz"
LIFTOVER_BIN = Path(os.environ["WORK"]) / "liftOver"
LIFTED_DIR  = Path(os.environ["WORK"]) / "g4_t2t_revisions" / "PRMD9_peaks_datasets_GSE166483"
LIFTED_DIR.mkdir(parents=True, exist_ok=True)

# ── Download chain if missing ─────────────────────────────────────────────
if not CHAIN_PATH.exists():
    print("Downloading hg38ToHs1 chain...")
    urllib.request.urlretrieve(
        "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/liftOver/hg38ToHs1.over.chain.gz",
        CHAIN_PATH
    )
    print("Done.")

PRMD9_hs1_files = {}
for allele, infile in PRMD9_infiles_hg38.items():
    out_file  = LIFTED_DIR / f"PRMD9_{allele}.peaks_lifted_to_hs1.tsv.gz"
    unmapped  = LIFTED_DIR / f"PRMD9_{allele}.unmapped.bed"
    # liftOver needs uncompressed input — decompress to temp file
    with tempfile.NamedTemporaryFile(suffix=".bed", delete=False) as tmp:
        tmp_path = Path(tmp.name)
        with gzip.open(infile, "rt") as f_in:
            tmp.write(f_in.read().encode())
    result = subprocess.run(
            ["liftOver", 
             str(tmp_path), 
             str(CHAIN_PATH), 
             "/dev/stdout", 
             str(unmapped)],
            capture_output=True, text=True
        )
    tmp_path.unlink()
    if result.returncode != 0:
        print(f"  {allele} ERROR: {result.stderr.strip()}")
        continue
    from io import StringIO
    lifted_df = pd.read_table(
        StringIO(result.stdout), 
        header=None,
        names=["seqID","start","end","score"]
    )
    lifted_df.to_csv(out_file, 
                     sep="\t", 
                     index=False, 
                     compression="gzip")
    PRMD9_hs1_files[allele] = out_file
    print(f"  {allele}: {len(lifted_df)} peaks lifted  ({unmapped.name} has unmapped)")
print(PRMD9_hs1_files)

## 4. Load eG4 data

In [ ]:
from tqdm import tqdm 

indir = Path(f"{os.getenv('SCRATCH')}/g4_t2t_revisions_data")
FASTA_hs1 = Path(f"{indir}/fasta/hs1.fa.gz")
assert FASTA_hs1.is_file()
# # # #
seq_sizes_hs1 = dict()
chrom_sequences_hs1 = dict()
genome_size_hs1 = 0
for seqID, seq in tqdm(parse_fasta(FASTA_hs1), total=24):
    chrom_sequences_hs1[seqID] = seq.upper()
    seq_sizes_hs1[seqID] = len(seq)
    genome_size_hs1 += seq_sizes_hs1[seqID]
    # # # #
genome_size_hs1

In [ ]:
dataset_path = SCRATCH.joinpath("data")
G4HUNTER = dataset_path / "pG4s_extractions" / "g4hunter" / "chm13v2_g4hunter.txt.gz"
REGEX = dataset_path / "pG4s_extractions" / "quadparser" / "chm13v2_regex_motifs.txt"
g4_df = pd.read_table(G4HUNTER)
regex_df = pd.read_table(REGEX)

datasets = [
    ("eG4", eg4_hs1_df),
    ("G4Hunter", g4_df),
    ("Quadparser", regex_df)
]

for name, df in datasets:
    print(f"{name} dataset:")
    print(df.head())

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyranges as pr
import joblib
from pybedtools import BedTool
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

COVERAGE_FIELDS = ["total_hits", "total_bases", "all_bases", "coverage"]
DEGREE = 2
CHUNK_SIZES = [50_000, 100_000, 250_000, 500_000, 1_000_000, 2_000_000, 5_000_000, 10_000_000]
GROUP, GROUP_DF = "G4Hunter", g4_df

MODELS_DIR = target_data / "gc_models_chunk_sweep"
MODELS_DIR.mkdir(exist_ok=True, parents=True)

def calculate_gw_density(df, genome_size):
    df_pr = pr.from_dict({
        "Chromosome": df["seqID"],
        "Start": df["start"],
        "End": df["end"]
    })
    df_pr_merged = df_pr.merge()
    total_bases = (df_pr_merged.End - df_pr_merged.Start).sum()
    return total_bases * 1e3 / genome_size


In [ ]:
def build_fit_data(chunk_size, min_frac=0.995):
    rows = []
    for seqID, seq in chrom_sequences_hs1.items():
        for s in range(0, len(seq), chunk_size):
            chunk = seq[s:s + chunk_size]
            total_gc = chunk.count("G") + chunk.count("C")
            total_ta = chunk.count("T") + chunk.count("A")
            adj = total_gc + total_ta
            if adj < min_frac * chunk_size:
                continue
            rows.append({
                "seqID": seqID,
                "start": s,
                "end": s + len(chunk),
                "total_gc": total_gc,
                "gc_prop": total_gc / adj,
            })
    return pd.DataFrame(rows)

def tag(chunk_size):
    return f"{chunk_size/1e3:g}kb" if chunk_size < 1_000_000 else f"{chunk_size/1e6:g}Mb"

gw_density = calculate_gw_density(GROUP_DF, genome_size_hs1)
motif_bed = BedTool.from_dataframe(GROUP_DF[["seqID", "start", "end"]]).sort().merge().sort()
sweep_models, sweep_residuals, sweep_train, summary = {}, {}, {}, []

for chunk_size in tqdm(CHUNK_SIZES, desc="chunk sizes"):
    fit_data = build_fit_data(chunk_size)
    if fit_data.empty:
        print(f"{tag(chunk_size)} -> no usable windows, skipped")
        continue
    fit_bed = BedTool.from_dataframe(fit_data)
    train_data = pd.read_table(
        fit_bed.coverage(motif_bed).fn,
        header=None,
        names=["seqID", "start", "end", "total_gc", "gc_prop"] + COVERAGE_FIELDS,
    )
    train_data["g4_density"] = train_data["total_bases"] * 1e3 / train_data["all_bases"]
    train_data["fold_enrichment"] = train_data["g4_density"] / gw_density
    train_data = train_data[train_data["fold_enrichment"] > 0].reset_index(drop=True).copy()

    X = train_data["gc_prop"].values.reshape(-1, 1)
    y = train_data["fold_enrichment"].values
    model = make_pipeline(PolynomialFeatures(degree=DEGREE), LinearRegression())
    model.fit(X, y)

    train_data["predicted_enrichment"] = model.predict(train_data[["gc_prop"]])
    train_data["residuals"] = train_data["fold_enrichment"] - train_data["predicted_enrichment"]
    resid = train_data["residuals"].values

    r2 = r2_score(y, model.predict(X))
    print(f"{GROUP}  |  interval = {tag(chunk_size):>6}  |  n = {len(train_data):>7,}  "
          f"|  R² (degree {DEGREE}) = {r2:.4f}  |  resid SD = {resid.std():.4f}")

    model_path = MODELS_DIR / f"{GROUP}_{tag(chunk_size)}_model.pkl"
    resid_path = MODELS_DIR / f"{GROUP}_{tag(chunk_size)}_residuals.npy"
    joblib.dump(model, model_path)
    np.save(resid_path, resid)

    sweep_models[chunk_size] = model
    sweep_residuals[chunk_size] = resid
    sweep_train[chunk_size] = train_data
    summary.append({
        "chunk_size": chunk_size,
        "chunk_mb": chunk_size / 1e6,
        "n_windows": len(train_data),
        "r2": r2,
        "resid_sd": resid.std(),
        "resid_iqr": np.subtract(*np.quantile(resid, [0.75, 0.25])),
        "mean_fe": train_data["fold_enrichment"].mean(),
        "model_path": str(model_path),
        "residuals_path": str(resid_path),
    })

summary = pd.DataFrame(summary)
summary.to_csv(MODELS_DIR / f"{GROUP}_chunk_sweep_summary.tsv", sep="\t", index=False)
print(f"\nSaved {len(sweep_models)} models + residuals to {MODELS_DIR}")
display(summary)

def load_sweep_models(models_dir, group=GROUP):
    loaded_models, loaded_residuals = {}, {}
    for model_path in sorted(models_dir.glob(f"{group}_*_model.pkl")):
        key = model_path.stem.replace(f"{group}_", "").replace("_model", "")
        resid_path = models_dir / f"{group}_{key}_residuals.npy"
        loaded_models[key] = joblib.load(model_path)
        if resid_path.exists():
            loaded_residuals[key] = np.load(resid_path)
    return loaded_models, loaded_residuals

n = len(sweep_train)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.4 * nrows), squeeze=False)
axes = axes.ravel()
for ax, (chunk_size, td) in zip(axes, sweep_train.items()):
    model = sweep_models[chunk_size]
    X = td["gc_prop"].values.reshape(-1, 1)
    r2 = r2_score(td["fold_enrichment"].values, model.predict(X))
    x_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
    ax.scatter(td["gc_prop"], td["fold_enrichment"], s=8, alpha=0.35, color="#4C72B0", lw=0)
    ax.plot(x_range, model.predict(x_range), color="red", lw=2)
    ax.text(0.03, 0.95,
            f"{tag(chunk_size)}\nn={len(td):,}\nR²={r2:.3f}",
            transform=ax.transAxes, va="top", ha="left", fontsize=13)
    ax.set_xlabel("GC proportion", fontsize=15)
    ax.set_ylabel("Fold enrichment", fontsize=15)
    ax.tick_params(axis="both", labelsize=13)
    ax.grid(lw=0.4, alpha=0.6)

for ax in axes[n:]:
    ax.axis("off")
fig.tight_layout()
plt.show()
fig.savefig(target_fig / f"{GROUP}_gc_fit_chunk_sweep.pdf", bbox_inches="tight", transparent=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].plot(summary["chunk_mb"], summary["r2"], marker="o", color="#4C72B0", lw=2)
axes[0].set_ylabel("R²", fontsize=15)
axes[1].plot(summary["chunk_mb"], summary["resid_sd"], marker="o", color="#C44E52", lw=2)
axes[1].set_ylabel("Residual SD (cleaned)", fontsize=15)
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("Training interval size (Mb)", fontsize=15)
    ax.tick_params(axis="both", labelsize=13)
    ax.grid(lw=0.4, alpha=0.6)
fig.tight_layout()
plt.show()
fig.savefig(target_fig / f"{GROUP}_gc_fit_chunk_sweep_summary.pdf", bbox_inches="tight", transparent=True)

fig, ax = plt.subplots(figsize=(7.5, 5))
for chunk_size, resid in sweep_residuals.items():
    sns.kdeplot(resid, ax=ax, lw=2, label=tag(chunk_size))
ax.set_xlabel("Residual (observed − predicted FE)", fontsize=15)
ax.set_ylabel("Density", fontsize=15)
ax.tick_params(axis="both", labelsize=13)
ax.grid(lw=0.4, alpha=0.6)
ax.legend(fontsize=11, frameon=False)
fig.tight_layout()
plt.show()
fig.savefig(target_fig / f"{GROUP}_gc_fit_chunk_sweep_residuals.pdf",
            bbox_inches="tight",
            transparent=True)


In [ ]:
# Create Models
%matplotlib inline
import seaborn as sns 
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import numpy as np
import pyranges as pr

COVERAGE_FIELDS = ["total_hits", "total_bases", "all_bases", "coverage"]
fit_data = []
CHUNK_SIZE = 2_000_000
NUCLEOTIDES = {"A", "G", "C", "T"}
datasets = [
            ("G4Hunter", g4_df), 
            ("Quadparser", regex_df),
            ("eG4", eg4_hs1_df)
            ]
models = dict()
residuals = dict()
rel_residuals = dict()
train_frames = dict()

def calculate_gw_density(df, genome_size):
    df_pr = pr.from_dict({
        "Chromosome": df["seqID"],
        "Start": df["start"],
        "End": df["end"]
    })
    df_pr_merged = df_pr.merge()
    total_bases = (df_pr_merged.End - df_pr_merged.Start).sum()
    gw_density = total_bases * 1e3 / genome_size    
    return gw_density

def star_me(observed_pval):
    vals = [0.0001, 0.001, 0.01, 0.05]
    for i, pval in zip([4, 3, 2, 1], vals):
        if observed_pval < pval:
            return "*" * i
    return "ns"

for seqID, seq in tqdm(chrom_sequences_hs1.items()):
    for s in range(0, len(seq), CHUNK_SIZE):
        chunk = seq[s:s+CHUNK_SIZE]
        total_gc = chunk.count("G") + chunk.count("C")
        total_ta = chunk.count("T") + chunk.count("A")
        CHUNK_SIZE_ADJ = total_gc + total_ta
        # CHUNK_SIZE_ADJ = sum(int(j in NUCLEOTIDES) for j in chunk)
        if CHUNK_SIZE_ADJ < CHUNK_SIZE - 10_000:
            continue
        gc_prop = total_gc / CHUNK_SIZE_ADJ
        fit_data.append({
                         "seqID": seqID,
                         "start": s,
                         "end": s + len(chunk),
                         "total_gc": total_gc,
                         "gc_prop": gc_prop
                         })
        
fit_data = pd.DataFrame(fit_data)
fit_data_bed = BedTool.from_dataframe(fit_data)

for group, df in datasets:
    gw_density = calculate_gw_density(df, genome_size_hs1)
    df_bed = BedTool.from_dataframe(df[["seqID", "start", "end"]]).sort().merge().sort()
    train_data = pd.read_table(fit_data_bed.coverage(df_bed).fn, 
                             header=None, 
                             names=["seqID", "start", "end", "total_gc", "gc_prop"] + COVERAGE_FIELDS)
    train_data.loc[:, "g4_density"] = train_data["total_bases"] * 1e3 / train_data["all_bases"]
    train_data.loc[:, "fold_enrichment"] = train_data["g4_density"] / gw_density
    train_data = train_data[train_data["fold_enrichment"] > 0].reset_index(drop=True).copy()
    X = train_data["gc_prop"].values.reshape(-1, 1)
    y = train_data["fold_enrichment"].values
    train_frames[group] = train_data

    # Polynomial regression (degree 2)
    degree = 2
    model = make_pipeline(
        PolynomialFeatures(degree=degree),
        LinearRegression()
    )
    model.fit(X, y)
    models[group] = model

    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    print(f"R² (degree {degree}): {r2:.4f}")

    train_data["predicted_enrichment"] = model.predict(train_data[["gc_prop"]])
    train_data.loc[:, "residuals"] = (train_data["fold_enrichment"] - train_data["predicted_enrichment"])
    residuals[group] = train_data["residuals"].values
    rel_residuals[group] = train_data["fold_enrichment"] / train_data["predicted_enrichment"] - 1

    # Plot with fit
    x_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
    y_range = model.predict(x_range)

    sns.scatterplot(data=train_data, 
                    y="fold_enrichment", 
                    x="gc_prop", 
                    alpha=0.5)
    plt.plot(x_range, 
            y_range, 
            color="red", 
            label=f"Poly degree {degree}, R²={r2:.3f}")
    plt.title(group, fontsize=16)
    plt.legend()
    plt.show()
    plt.gcf().savefig(target_fig / f"{group}_gc_fit.pdf", 
                      bbox_inches="tight", 
                      transparent=True)

    # --- minimum reliable bin size ---
    p_g4 = train_data["total_bases"].sum() / train_data["all_bases"].sum()
    print(f"\n{group}  |  p_g4 = {p_g4:.5f}  |  gw_density = {gw_density:.4f} per kb")
    print(f"{'CV threshold':<14} {'n_min':>12}")
    for eps in [0.10, 0.05, 0.02, 0.01]:
        n_min = 1 / (p_g4 * eps**2)
        print(f"  CV < {eps:.0%}        {n_min/1e3:>8.0f} kb")
    print()

import joblib
MODELS_DIR = target_data / "gc_models"
MODELS_DIR.mkdir(exist_ok=True, parents=True)

for group, model in models.items():
    joblib.dump(model, MODELS_DIR / f"{group}_model.pkl")
for group, resid in residuals.items():
    np.save(MODELS_DIR / f"{group}_residuals.npy", resid)
for group, rel_resid in rel_residuals.items():
    np.save(MODELS_DIR / f"{group}_rel_residuals.npy", rel_resid)

print(f"Saved models and residuals to {MODELS_DIR}")

In [ ]:
def calculate_gw_density(df, genome_size):
    df_pr = pr.from_dict({
        "Chromosome": df["seqID"],
        "Start": df["start"],
        "End": df["end"]
    })
    df_pr_merged = df_pr.merge()
    total_bases = (df_pr_merged.End - df_pr_merged.Start).sum()
    gw_density = total_bases * 1e3 / genome_size    
    return gw_density
gw_densities = dict()
for group, df in datasets:
    gw_densities[group] = calculate_gw_density(df, genome_size_hs1)

In [ ]:
alleles = hotspots_hs1.keys()
hotspots_merged_df = dict()
for allele in alleles:
    temp_df = pr.PyRanges(hotspots_hs1[allele][["seqID", "start", "end"]].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}))\
                            .merge()\
                            .as_df()\
                                .rename(columns={"Chromosome": "seqID", "Start": "start", "End": "end"})
    temp_df.loc[:, "sequence"] = temp_df.apply(lambda row: chrom_sequences_hs1[row["seqID"]][row["start"]:row["end"]], axis=1)
    temp_df.loc[:, "GC_content"] = temp_df["sequence"].str.count("G|C")
    temp_df.loc[:, "allele"] = allele
    # temp_df.loc[:, "motif_id"] = allele
    hotspots_merged_df[allele] = temp_df

hotspots_merged_df = pd.concat(hotspots_merged_df.values(), ignore_index=True)
hotspots_merged_df

In [ ]:
hotspots_bed = BedTool.from_dataframe(hotspots_merged_df[["seqID", "start", "end", "GC_content", "allele"]].copy()).sort()

In [ ]:
import polars as pl

def empirical_pvalue(null, obs, alternative="two-sided"):
    null = np.asarray(null)
    n = len(null)
    p_hi = (1 + (null >= obs).sum()) / (1 + n)
    p_lo = (1 + (null <= obs).sum()) / (1 + n)
    if alternative == "greater":
        return p_hi
    if alternative == "less":
        return p_lo
    return min(2 * min(p_hi, p_lo), 1.0)
datasets = [
            ("G4Hunter", g4_df), 
            ("Quadparser", regex_df), 
            ("eG4", eg4_hs1_df)
            ]
datasets_bed = {}
for group, df in datasets:
    df_bed = BedTool.from_dataframe(df[["seqID", "start", "end"]]).sort().merge().sort()
    datasets_bed[group] = df_bed
results_hotspots_df = []
for group, df_bed in datasets_bed.items():
    if group.startswith("G4Hunter"):
        model_name = "G4Hunter" 
    else:
        model_name = group
    overlap_df = (
            pl.read_csv(
                        hotspots_bed.coverage(df_bed).fn,
                        has_header=False,
                        separator="\t",
                        new_columns=["chrom", "start", "end", "GC_content", "allele", "total_hits", "total_bases", "all_bases", "coverage"]
            )
            .with_columns(
                at_least_one = (pl.col("total_hits") > 0).cast(pl.Int64)
            )
            .group_by(["allele"])
            .agg(
                    pl.col("all_bases").sum(),
                    pl.col("total_bases").sum(),
                    pl.col("at_least_one").sum().alias("occupancy"),
                    pl.col("all_bases").count().alias("total_peaks"),
                    (1e2 * pl.col("at_least_one").mean()).alias("at_least_one"),
                    pl.col("GC_content").sum(),
            )
            .with_columns(
                g4_density=1e3 * pl.col("total_bases") / pl.col("all_bases"),
                GC_prop=pl.col("GC_content") / pl.col("all_bases") 
            )
            .with_columns(
                fold_enrichment=pl.col("g4_density") / gw_densities[group]
            )
            .with_columns(
                group=pl.lit(group)
            )
    )

    predicted_GC_content = models[model_name].predict(overlap_df["GC_prop"].to_numpy().reshape(-1, 1))
    overlap_df = (
            overlap_df.with_columns(
                predicted_enrichment=predicted_GC_content
            )
            .with_columns(
                residuals=pl.col("fold_enrichment") - pl.col("predicted_enrichment")
            )
            .with_columns(
                rel_enrichment=pl.col("fold_enrichment") / pl.col("predicted_enrichment"),
            )
    )
    overlap_df = overlap_df.with_columns(
        significant=pl.col("residuals").map_elements(lambda x: empirical_pvalue(residuals[model_name], x, alternative="greater"),
        return_dtype=pl.Float64),
    ) 
    results_hotspots_df.append(overlap_df)
# # #
results_hotspots_df = pl.concat(results_hotspots_df)
pvalues = multipletests(results_hotspots_df["significant"], method="fdr_bh")
results_hotspots_df = results_hotspots_df.with_columns(
    significant_adj=pvalues[1],
    significant=pvalues[0],
)\
    .with_columns(
        stars=pl.col("significant_adj").map_elements(star_me, return_dtype=pl.Utf8),
    )
results_hotspots_df 

In [ ]:
import polars as pl
PRMD9_bed = BedTool.from_dataframe(merged_PRMD9_df[["chrom", "start", "end", "GC_content", "motif_id"]]).sort()
datasets_bed = {}
for group, df in datasets:
    df_bed = BedTool.from_dataframe(df[["seqID", "start", "end"]]).sort().merge().sort()
    datasets_bed[group] = df_bed

results_df = []
for group, df_bed in datasets_bed.items():
    overlap_df = (
            pl.read_csv(
                        PRMD9_bed.coverage(df_bed).fn,
                        has_header=False,
                        separator="\t",
                        new_columns=["chrom", "start", "end", "GC_content", "motif_id", "total_hits", "total_bases", "all_bases", "coverage"]
                )
            .with_columns(
                at_least_one = (pl.col("total_hits") > 0).cast(pl.Int64)
            )
            .group_by(["motif_id"])
            .agg(
                    pl.col("all_bases").sum(),
                    pl.col("total_bases").sum(),
                    (1e2 * pl.col("at_least_one").mean()).alias("at_least_one"),
                    pl.col("GC_content").sum(),
            )
            .with_columns(
                g4_density=1e3 * pl.col("total_bases") / pl.col("all_bases"),
                GC_prop=pl.col("GC_content") / pl.col("all_bases") 
            )
            .with_columns(
                fold_enrichment=pl.col("g4_density") / gw_densities[group]
            )
            .with_columns(
                group=pl.lit(group)
            )
    )
    predicted_GC_content = models[group].predict(overlap_df["GC_prop"].to_numpy().reshape(-1, 1))
    overlap_df = (
            overlap_df.with_columns(
                predicted_enrichment=predicted_GC_content
            )
            .with_columns(
                residuals=pl.col("fold_enrichment") - pl.col("predicted_enrichment")
            )
            .with_columns(
                rel_residuals=pl.col("fold_enrichment") / pl.col("predicted_enrichment") - 1
            )
    )
    overlap_df = overlap_df.with_columns(
        significant=pl.col("residuals").map_elements(lambda x: empirical_pvalue(residuals[group], x), return_dtype=pl.Float64),
        significant_rel=pl.col("rel_residuals").map_elements(lambda x: empirical_pvalue(rel_residuals[group], x),return_dtype=pl.Float64)
    ) 
    results_df.append(overlap_df)
results_df = pl.concat(results_df)

pvalues = multipletests(results_df["significant"], method="fdr_bh")
pvalues_rel = multipletests(results_df["significant_rel"], method="fdr_bh")
results_df = results_df.with_columns(
    significant_adj=pvalues[1],
    significant=pvalues[0],
    significant_rel_adj=pvalues_rel[1],
    significant_rel=pvalues_rel[0]
)\
    .with_columns(
        stars=pl.col("significant_adj").map_elements(star_me, return_dtype=pl.Utf8),
        stars_rel=pl.col("significant_rel_adj").map_elements(star_me, return_dtype=pl.Utf8)
    )
results_df 

In [ ]:
results_hotspots_df.filter(pl.col("group") == "G4Hunter")[["allele", "at_least_one"]]

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

palette   = {"Quadparser": "#6272d4", "G4Hunter": "#2166ac", "eG4": "#ded714"}
BAR_W     = 0.25
groups    = ["G4Hunter", "Quadparser", "eG4"]
df_perc         = results_hotspots_df.to_pandas()
allele_order_perc = (df_perc[df_perc["group"] == "G4Hunter"]
                     .sort_values("at_least_one", ascending=False)["allele"].tolist())
fig, ax = plt.subplots(figsize=(12, 5))
x       = np.arange(len(allele_order_perc))
offsets = np.linspace(-(len(groups) - 1) / 2, (len(groups) - 1) / 2, len(groups)) * BAR_W
for j, group in enumerate(groups):
    sub  = df_perc[df_perc["group"] == group].set_index("allele").reindex(allele_order_perc)
    vals = sub["at_least_one"].values
    ax.bar(x + offsets[j], vals, width=BAR_W, color=palette[group],
           edgecolor="black", linewidth=1.5, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(allele_order_perc, fontsize=22)
ax.set_ylabel("Peaks with ≥1 G4 (%)", fontsize=22)
ax.tick_params(axis="both", labelsize=18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", lw=0.4, alpha=0.6)
ax.set_axisbelow(True)
handles = [mpatches.Patch(color=palette[g], label=g) for g in groups]
ax.legend(handles=handles, fontsize=18, frameon=True, fancybox=True,
          bbox_to_anchor=(1.0, 1.0))

fig.savefig(f"{target_fig}/prmd9_g4_peaks_pct.pdf", 
            dpi=600, 
            transparent=True, 
            bbox_inches="tight")
plt.show()
plt.close()


## 5. Load hs1 genome

In [ ]:
seq_sizes_hs1      = {}
chrom_sequences_hs1 = {}
genome_size_hs1    = 0

for seqID, seq in tqdm(parse_fasta(FASTA_HS1), total=24):
    chrom_sequences_hs1[seqID] = seq.upper()
    seq_sizes_hs1[seqID]       = len(seq)
    genome_size_hs1           += len(seq)

print(f"Genome size: {genome_size_hs1:,} bp")

## 6. GC-matched paired control generation

In [ ]:
MAX_TRIES = 5_000
GC_TOL    = 0.01

def seq_gc(seq):
    gc  = seq.count("G") + seq.count("C")
    tot = len(seq) - seq.count("N")
    return gc / tot if tot > 0 else 0.0

def build_blacklist(hotspot_dfs):
    """Index all hotspot intervals per chromosome for fast overlap checking."""
    all_peaks = hotspot_dfs.drop_duplicates()
    # all_peaks = pd.concat(hotspot_dfs.values()).drop_duplicates()
    index = {}
    for chrom, grp in all_peaks.groupby("chrom"):
        index[chrom] = sorted(zip(grp["start"], grp["end"]))
    return index

def overlaps_any(chrom, start, end, index):
    if chrom not in index:
        return False
    for bl_s, bl_e in index[chrom]:
        if bl_s >= end:
            break
        if bl_e > start:
            return True
    return False

def sample_control(length, target_gc, chrom, blacklist, local_bl):
    chrom_len = seq_sizes_hs1.get(chrom, 0)
    if chrom_len < length:
        return None
    for _ in range(MAX_TRIES):
        start = random.randint(0, chrom_len - length)
        end   = start + length
        if overlaps_any(chrom, start, end, blacklist):
            continue
        if overlaps_any(chrom, start, end, local_bl):
            continue
        if abs(seq_gc(chrom_sequences_hs1[chrom][start:end]) - target_gc) <= GC_TOL:
            return {"seqID": chrom, "start": start, "end": end}
    return None

def generate_controls(peaks_df, blacklist, seed=42):
    random.seed(seed)
    local_bl = {}
    records, unmatched = [], 0
    for _, peak in peaks_df.iterrows():
        chrom     = peak["seqID"]
        length    = int(peak["end"] - peak["start"])
        if chrom not in chrom_sequences_hs1:
            unmatched += 1
            continue
        target_gc = seq_gc(chrom_sequences_hs1[chrom][int(peak["start"]):int(peak["end"])])
        ctrl = sample_control(length, target_gc, chrom, blacklist, local_bl)
        if ctrl:
            records.append({
                "peak_seqID": chrom,
                "peak_start": int(peak["start"]),
                "peak_end":   int(peak["end"]),
                "ctrl_seqID": ctrl["seqID"],
                "ctrl_start": ctrl["start"],
                "ctrl_end":   ctrl["end"],
            })
            local_bl.setdefault(chrom, [])
            bisect.insort(local_bl[chrom], (ctrl["start"], ctrl["end"]))
        else:
            unmatched += 1
    print(f"  matched {len(records)}/{len(peaks_df)}  unmatched {unmatched}")
    return pd.DataFrame(records)

def load_or_generate_controls(hotspots_hs1, blacklist_df, ctrl_out, force=False):
    ctrl_out.mkdir(parents=True, exist_ok=True)
    blacklist = build_blacklist(blacklist_df)
    controls  = {}
    for allele, df in hotspots_hs1.items():
        cache = ctrl_out / f"hotspot_{allele}_controls_with_motifs.tsv.gz"
        if cache.exists() and not force:
            ctrl_df = pd.read_table(cache)
            print(f"  {allele}: loaded from cache ({len(ctrl_df):,} pairs)")
        else:
            print(f"── {allele} ──")
            ctrl_df = generate_controls(df, blacklist)
            ctrl_df.to_csv(cache, sep="\t", index=False, compression="gzip")
        controls[allele] = ctrl_df
    return controls

In [ ]:
blacklist_df = pd.concat([hotspots_hs1[allele][["seqID", "start", "end"]] for allele in ALLELE_COLS], ignore_index=True)
blacklist_df = (
    pr.PyRanges(blacklist_df.rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}))
    .merge()
    .as_df()
    .rename(columns={"Chromosome": "chrom", "Start": "start", "End": "end"})
    .sort_values(["chrom", "start"])
    .reset_index(drop=True)
)
blacklist_df


In [ ]:
def load_or_generate_controls(hotspots_hs1, blacklist_df, ctrl_out, force=False):
    ctrl_out.mkdir(parents=True, exist_ok=True)
    blacklist = build_blacklist(blacklist_df)
    controls  = {}
    for allele, df in hotspots_hs1.items():
        cache = ctrl_out / f"hotspot_{allele}_controls_with_motifs.tsv.gz"
        if cache.exists() and not force:
            ctrl_df = pd.read_table(cache)
            print(f"  {allele}: loaded from cache ({len(ctrl_df):,} pairs)")
        else:
            print(f"── {allele} ──")
            ctrl_df = generate_controls(df, blacklist)
            ctrl_df.to_csv(cache, sep="\t", index=False, compression="gzip")
        controls[allele] = ctrl_df
    return controls

FORCE = False
hotspot_controls = load_or_generate_controls(hotspots_hs1, blacklist_df, target_data, force=FORCE)
hotspot_controls["AA1"].head()

In [ ]:
hotspots_hs1_merged = dict()
for allele, df in hotspots_hs1.items():
    temp_df = (
        pr.PyRanges(df[["seqID", "start", "end"]].rename(columns={"seqID": "Chromosome", "start": "Start", "end": "End"}))
        .merge()
        .as_df()
        .rename(columns={"Chromosome": "seqID", "Start": "start", "End": "end"})
    )
    temp_df.loc[:, "sequence"] = temp_df.apply(lambda row: chrom_sequences_hs1[row["seqID"]][row["start"]:row["end"]], axis=1)
    temp_df.loc[:, "GC_content"] = temp_df["sequence"].str.count("G|C")
    temp_df.loc[:, "allele"] = allele
    hotspots_hs1_merged[allele] = temp_df

In [ ]:
def load_or_generate_controls(hotspots_hs1, blacklist_df, ctrl_out, force=False):
    ctrl_out.mkdir(parents=True, exist_ok=True)
    blacklist = build_blacklist(blacklist_df)
    controls  = {}
    for allele, df in hotspots_hs1.items():
        cache = ctrl_out / f"hotspot_{allele}_controls_with_motifs.merged.tsv.gz"
        if cache.exists() and not force:
            ctrl_df = pd.read_table(cache)
            print(f"  {allele}: loaded from cache ({len(ctrl_df):,} pairs)")
        else:
            print(f"── {allele} ──")
            ctrl_df = generate_controls(df, blacklist)
            ctrl_df.to_csv(cache, sep="\t", index=False, compression="gzip")
        controls[allele] = ctrl_df
    return controls

FORCE = False
hotspot_controls_merged = load_or_generate_controls(hotspots_hs1_merged, blacklist_df, target_data, force=FORCE)
hotspot_controls_merged["AA1"].head()

In [ ]:
def verify_no_overlaps(hotspots_hs1, hotspot_controls):
    # Build one BED of all hotspots across all alleles
    all_hotspots_bed = BedTool.from_dataframe(
        pd.concat(hotspots_hs1.values())
          .rename(columns={"seqID": "seqID"})  # already named correctly
          [["seqID", "start", "end"]]
          .drop_duplicates()
    ).sort()

    print("=== Hotspot × Control overlaps ===")
    for allele, pairs_df in hotspot_controls.items():
        ctrl_bed = BedTool.from_dataframe(
            pairs_df[["ctrl_seqID", "ctrl_start", "ctrl_end"]]
            .rename(columns={"ctrl_seqID": "seqID", "ctrl_start": "start", "ctrl_end": "end"})
        ).sort()

        n_overlaps = ctrl_bed.intersect(all_hotspots_bed, u=True).count()
        print(f"  {allele}: {n_overlaps} controls overlap a hotspot  {'✓' if n_overlaps == 0 else '✗ PROBLEM'}")

    print("\n=== Control × Control overlaps (within allele) ===")
    for allele, pairs_df in hotspot_controls.items():
        ctrl_bed = BedTool.from_dataframe(
            pairs_df[["ctrl_seqID", "ctrl_start", "ctrl_end"]]
            .rename(columns={"ctrl_seqID": "seqID", "ctrl_start": "start", "ctrl_end": "end"})
        ).sort()

        # intersect with itself, exclude self-overlaps
        n_overlaps = ctrl_bed.intersect(ctrl_bed, u=True, f=1e-9).count() - len(pairs_df)
        print(f"  {allele}: {max(0, n_overlaps)} control-control overlaps  {'✓' if n_overlaps <= 0 else '✗ PROBLEM'}")

verify_no_overlaps(hotspots_hs1, hotspot_controls)
print("MERGED")
verify_no_overlaps(hotspots_hs1_merged, hotspot_controls_merged)


In [ ]:
from scipy.stats import ks_2samp, mannwhitneyu

allele = "AA1"
pairs  = hotspot_controls[allele]
peak_gc = [
    seq_gc(chrom_sequences_hs1[r.peak_seqID][r.peak_start:r.peak_end])
    for r in pairs.itertuples() if r.peak_seqID in chrom_sequences_hs1
]
ctrl_gc = [
    seq_gc(chrom_sequences_hs1[r.ctrl_seqID][r.ctrl_start:r.ctrl_end])
    for r in pairs.itertuples() if r.ctrl_seqID in chrom_sequences_hs1
]

ks_stat, ks_p   = ks_2samp(peak_gc, ctrl_gc)
mw_stat, mw_p   = mannwhitneyu(peak_gc, ctrl_gc, alternative="two-sided")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(peak_gc, bins=40, alpha=0.6, label="Hotspot", color="#2166ac")
ax.hist(ctrl_gc, bins=40, alpha=0.6, label="Control", color="#d73027")
ax.set_xlabel("GC content", fontsize=14)
ax.set_ylabel("Count", fontsize=14)
ax.legend(fontsize=12)
ax.grid(lw=0.4, alpha=0.6)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Mean  — peak: {np.mean(peak_gc):.4f}  ctrl: {np.mean(ctrl_gc):.4f}")
print(f"KS    — stat={ks_stat:.4f}  p={ks_p:.4f}  {'✓ same distribution' if ks_p > 0.05 else '✗ distributions differ'}")
print(f"MWU   — stat={mw_stat:.0f}  p={mw_p:.4f}  {'✓ same distribution' if mw_p > 0.05 else '✗ distributions differ'}")

In [ ]:
from scipy.stats import ks_2samp, mannwhitneyu

allele = "AA1"
pairs  = hotspot_controls_merged[allele]
peak_gc = [
    seq_gc(chrom_sequences_hs1[r.peak_seqID][r.peak_start:r.peak_end])
    for r in pairs.itertuples() if r.peak_seqID in chrom_sequences_hs1
]
ctrl_gc = [
    seq_gc(chrom_sequences_hs1[r.ctrl_seqID][r.ctrl_start:r.ctrl_end])
    for r in pairs.itertuples() if r.ctrl_seqID in chrom_sequences_hs1
]

ks_stat, ks_p   = ks_2samp(peak_gc, ctrl_gc)
mw_stat, mw_p   = mannwhitneyu(peak_gc, ctrl_gc, alternative="two-sided")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(peak_gc, bins=40, alpha=0.6, label="Hotspot", color="#2166ac")
ax.hist(ctrl_gc, bins=40, alpha=0.6, label="Control", color="#d73027")
ax.set_xlabel("GC content", fontsize=14)
ax.set_ylabel("Count", fontsize=14)
ax.legend(fontsize=12)
ax.grid(lw=0.4, alpha=0.6)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

print(f"Mean  — peak: {np.mean(peak_gc):.4f}  ctrl: {np.mean(ctrl_gc):.4f}")
print(f"KS    — stat={ks_stat:.4f}  p={ks_p:.4f}  {'✓ same distribution' if ks_p > 0.05 else '✗ distributions differ'}")
print(f"MWU   — stat={mw_stat:.0f}  p={mw_p:.4f}  {'✓ same distribution' if mw_p > 0.05 else '✗ distributions differ'}")

## 7. Coverage and enrichment analysis

In [ ]:
def region_coverage(pairs_df, g4_bed):
    """Compute per-pair coverage fractions. Row order preserved — critical for paired Wilcoxon."""
    peak_bed = BedTool.from_dataframe(
        pairs_df[["peak_seqID", "peak_start", "peak_end"]].rename(
            columns={"peak_seqID": "seqID", 
                     "peak_start": "start", 
                     "peak_end": "end"})
    )  # NO .sort() — preserves row order for paired Wilcoxon
    ctrl_bed = BedTool.from_dataframe(
        pairs_df[["ctrl_seqID", "ctrl_start", "ctrl_end"]].rename(
            columns={"ctrl_seqID": "seqID", "ctrl_start": "start", "ctrl_end": "end"})
    )
    def cov(bed):
        return bed.coverage(g4_bed).to_dataframe(
            names=["chrom", "start", "end", "num_overlaps", "bases_covered", "reg_len", "frac"])
    cov_g4   = cov(peak_bed)
    cov_ctrl = cov(ctrl_bed)
    g4_dens   = cov_g4["bases_covered"].sum()   * 1e3 / cov_g4["reg_len"].sum()
    ctrl_dens = cov_ctrl["bases_covered"].sum()  * 1e3 / cov_ctrl["reg_len"].sum()
    g4_yes    = (cov_g4["num_overlaps"]   > 0).sum()
    ct_yes    = (cov_ctrl["num_overlaps"] > 0).sum()
    n         = len(cov_g4)
    return g4_dens, ctrl_dens, g4_yes, ct_yes, n, cov_g4["frac"].values, cov_ctrl["frac"].values

def permutation_test(g4_fracs, ctrl_fracs, n_perm=10_000, seed=0):
    rng   = np.random.default_rng(seed)
    diffs = g4_fracs - ctrl_fracs
    obs   = diffs.mean()
    signs      = rng.choice([-1, 1], size=(n_perm, len(diffs)))
    perm_stats = (signs * diffs).mean(axis=1)
    pval       = (np.sum(np.abs(perm_stats) >= np.abs(obs)) + 1) / (n_perm + 1)
    return obs, pval

def starme(pval):
    for thresh, label in [(1e-4, "***"), (1e-3, "**"), (0.01, "*"), (0.05, ".")]:
        if pval < thresh:
            return label
    return "ns"

In [ ]:
datasets = [
    ("eG4", eg4_hs1_df),
    ("G4Hunter", g4_df),
    ("Quadparser", regex_df)
]
datasets_bed = {name: BedTool.from_dataframe(df[["seqID", "start", "end"]]).sort() for name, df in datasets}

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import norm

results = []
for allele, pairs_df in hotspot_controls.items():
    print(f"── {allele} ({len(pairs_df):,} pairs) ──")
    for group, df_bed in datasets_bed.items():
        g4_dens, ctrl_dens, g4_yes, ct_yes, g4_n, g4_fracs, ctrl_fracs = region_coverage(
            pairs_df, df_bed
        )
        rel_fe = g4_dens / ctrl_dens if ctrl_dens > 0 else np.nan
        diffs    = g4_fracs - ctrl_fracs
        nz       = diffs != 0
        cliffs_d = np.mean(np.sign(diffs[nz])) if nz.sum() else np.nan
        peak_hit = g4_fracs   > 0
        ctrl_hit = ctrl_fracs > 0
        a = int(np.sum(peak_hit  &  ctrl_hit))  
        b = int(np.sum(peak_hit  & ~ctrl_hit)) 
        c = int(np.sum(~peak_hit &  ctrl_hit))  
        d = int(np.sum(~peak_hit & ~ctrl_hit))

        res_mcn      = mcnemar([[a, b], [c, d]], exact=True)
        pval_mcnemar = res_mcn.pvalue
        or_mcnemar   = b / c if c > 0 else (np.inf if b > 0 else np.nan)

        # Wald CI on log-OR; Haldane 0.5 correction only if a cell is empty
        bh, ch = (b, c) if (b > 0 and c > 0) else (b + 0.5, c + 0.5)
        log_or       = np.log(bh / ch)
        se_log_or    = np.sqrt(1 / bh + 1 / ch)
        z            = log_or / se_log_or
        or_lo, or_hi = np.exp(log_or - 1.96 * se_log_or), np.exp(log_or + 1.96 * se_log_or)

        # exact p underflows to 0 at these counts — fall back to the z approximation
        pval_z      = 2 * norm.sf(abs(z))
        pval_report = pval_mcnemar if pval_mcnemar > 0 else pval_z

        _, pval_wx   = wilcoxon(g4_fracs, ctrl_fracs, zero_method="zsplit", alternative="two-sided")
        _, pval_perm = permutation_test(g4_fracs, ctrl_fracs)

        results.append({
            "allele":        allele,
            "g4_density":    g4_dens,
            "ctrl_density":  ctrl_dens,
            "rel_fe":        rel_fe,
            "cliffs_delta":  cliffs_d,
            "n_nontied":     int(nz.sum()),
            "g4_yes":        g4_yes,
            "ct_yes":        ct_yes,
            "g4_n":          g4_n,
            "occ_peak_pct":  100 * g4_yes / g4_n,
            "occ_ctrl_pct":  100 * ct_yes / g4_n,
            "disc_b":        b,
            "disc_c":        c,
            "or_mcnemar":    or_mcnemar,
            "or_lo":         or_lo,
            "or_hi":         or_hi,
            "log_or":        log_or,
            "se_log_or":     se_log_or,
            "z_mcnemar":     z,
            "pval_mcnemar":  pval_mcnemar,
            "pval_z":        pval_z,
            "pval_report":   pval_report,
            "exact_underflow": pval_mcnemar == 0,
            "pval_wx":       max(pval_wx,   np.finfo(float).tiny),
            "pval_perm":     max(pval_perm, np.finfo(float).tiny),
            "group": group,
        })
        print(f"{group}:  rel_fe={rel_fe:.4f}  delta={cliffs_d:.3f}  "
              f"occ={100*g4_yes/g4_n:.1f}% vs {100*ct_yes/g4_n:.1f}%  "
              f"OR={or_mcnemar:.2f} [{or_lo:.2f}, {or_hi:.2f}]  z={z:.1f}  "
              f"(b={b}, c={c})  p={pval_report:.2e}")

results_df = pd.DataFrame(results)
results_df

In [ ]:
for col in ["pval_mcnemar", "pval_wx", "pval_perm"]:
    adj_col   = col.replace("pval", "adj_pval")
    stars_col = col.replace("pval", "stars")
    _, results_df[adj_col], _, _ = multipletests(results_df[col], method="fdr_bh")
    results_df[stars_col] = results_df[adj_col].map(starme)

results_df.to_csv(target_data / "prmd9_hotspot_data.tsv", sep="\t", index=False)

In [ ]:
results_df.query("group == 'eG4'")[["allele", "rel_fe", "adj_pval_perm"]]

In [ ]:
results_df.query("group == 'G4Hunter'")[["allele", "rel_fe"]]

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import norm

results_merged = []
for allele, pairs_df in hotspot_controls_merged.items():
    print(f"── {allele} ({len(pairs_df):,} pairs) ──")
    for group, df_bed in datasets_bed.items():
        g4_dens, ctrl_dens, g4_yes, ct_yes, g4_n, g4_fracs, ctrl_fracs = region_coverage(
            pairs_df, df_bed
        )
        rel_fe = g4_dens / ctrl_dens if ctrl_dens > 0 else np.nan
        diffs    = g4_fracs - ctrl_fracs
        nz       = diffs != 0
        cliffs_d = np.mean(np.sign(diffs[nz])) if nz.sum() else np.nan
        peak_hit = g4_fracs   > 0
        ctrl_hit = ctrl_fracs > 0
        a = int(np.sum(peak_hit  &  ctrl_hit))  
        b = int(np.sum(peak_hit  & ~ctrl_hit)) 
        c = int(np.sum(~peak_hit &  ctrl_hit))  
        d = int(np.sum(~peak_hit & ~ctrl_hit))

        res_mcn      = mcnemar([[a, b], [c, d]], exact=True)
        pval_mcnemar = res_mcn.pvalue
        or_mcnemar   = b / c if c > 0 else (np.inf if b > 0 else np.nan)

        # Wald CI on log-OR; Haldane 0.5 correction only if a cell is empty
        bh, ch = (b, c) if (b > 0 and c > 0) else (b + 0.5, c + 0.5)
        log_or       = np.log(bh / ch)
        se_log_or    = np.sqrt(1 / bh + 1 / ch)
        z            = log_or / se_log_or
        or_lo, or_hi = np.exp(log_or - 1.96 * se_log_or), np.exp(log_or + 1.96 * se_log_or)

        # exact p underflows to 0 at these counts — fall back to the z approximation
        pval_z      = 2 * norm.sf(abs(z))
        pval_report = pval_mcnemar if pval_mcnemar > 0 else pval_z

        _, pval_wx   = wilcoxon(g4_fracs, ctrl_fracs, zero_method="zsplit",alternative="two-sided")
        _, pval_perm = permutation_test(g4_fracs, ctrl_fracs)

        results_merged.append({
            "allele":        allele,
            "g4_density":    g4_dens,
            "ctrl_density":  ctrl_dens,
            "rel_fe":        rel_fe,
            "cliffs_delta":  cliffs_d,
            "n_nontied":     int(nz.sum()),
            "g4_yes":        g4_yes,
            "ct_yes":        ct_yes,
            "g4_n":          g4_n,
            "occ_peak_pct":  100 * g4_yes / g4_n,
            "occ_ctrl_pct":  100 * ct_yes / g4_n,
            "disc_b":        b,
            "disc_c":        c,
            "or_mcnemar":    or_mcnemar,
            "or_lo":         or_lo,
            "or_hi":         or_hi,
            "log_or":        log_or,
            "se_log_or":     se_log_or,
            "z_mcnemar":     z,
            "pval_mcnemar":  pval_mcnemar,
            "pval_z":        pval_z,
            "pval_report":   pval_report,
            "exact_underflow": pval_mcnemar == 0,
            "pval_wx":       max(pval_wx,   np.finfo(float).tiny),
            "pval_perm":     max(pval_perm, np.finfo(float).tiny),
            "group": group,
        })
        print(f"{group}:  rel_fe={rel_fe:.4f}  delta={cliffs_d:.3f}  "
              f"occ={100*g4_yes/g4_n:.1f}% vs {100*ct_yes/g4_n:.1f}%  "
              f"OR={or_mcnemar:.2f} [{or_lo:.2f}, {or_hi:.2f}]  z={z:.1f}  "
              f"(b={b}, c={c})  p={pval_report:.2e}")

results_merged_df = pd.DataFrame(results_merged)
results_merged_df

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

RESULTS = results_df          # or results_merged_df / results_hotspots_df
OR_COL = "or_mcnemar"
STARS_COL = "stars_mcnemar"   # created in HS:52; falls back to raw p if absent

PEAK_COLOR = "#2166ac"
CTRL_COLOR = "#c8ccd4"
BAR_W = 0.38
STAR_FS, OR_FS = 15, 13
groups = ["G4Hunter", "Quadparser", "eG4"]

df = RESULTS.to_pandas() if hasattr(RESULTS, "to_pandas") else RESULTS.copy()

if STARS_COL not in df.columns:
    def starme(p):
        return "****" if p < 1e-4 else "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "ns"
    df[STARS_COL] = df["pval_report"].map(starme)

allele_order = (
    df[df["group"] == "G4Hunter"].sort_values("occ_peak_pct", ascending=False)["allele"].tolist()
)

fig, axes = plt.subplots(1, len(groups), figsize=(21, 6), sharey=True)
x = np.arange(len(allele_order))

for ax, group in zip(axes, groups):
    sub = df[df["group"] == group].set_index("allele").reindex(allele_order)
    peak = sub["occ_peak_pct"].values
    ctrl = sub["occ_ctrl_pct"].values

    ax.bar(x - BAR_W / 2, peak, width=BAR_W, color=PEAK_COLOR,
           edgecolor="black", linewidth=1.5, zorder=3)
    ax.bar(x + BAR_W / 2, ctrl, width=BAR_W, color=CTRL_COLOR,
           edgecolor="black", linewidth=1.5, zorder=3)

    span = max(np.nanmax(peak), np.nanmax(ctrl))
    pad, tick = 0.035 * span, 0.018 * span

    for i, allele in enumerate(allele_order):
        x1, x2 = x[i] - BAR_W / 2, x[i] + BAR_W / 2
        y = max(peak[i], ctrl[i]) + pad
        ax.plot([x1, x1, x2, x2], [y, y + tick, y + tick, y],
                lw=1.4, c="black", clip_on=False, zorder=4)

        stars = sub[STARS_COL].iloc[i]
        ax.annotate(stars, xy=(x[i], y + tick), xytext=(0, 4),
                    textcoords="offset points", ha="center", va="bottom",
                    rotation=90, fontsize=STAR_FS, clip_on=False)
        # stack the OR clear of the rotated stars: their height scales with length
        or_off = 4 + len(stars) * STAR_FS * 0.62 + 7
        ax.annotate(f"OR={sub[OR_COL].iloc[i]:.2f}", xy=(x[i], y + tick),
                    xytext=(0, or_off), textcoords="offset points",
                    ha="center", va="bottom", fontsize=OR_FS, clip_on=False)

    ax.set_xticks(x)
    ax.set_xticklabels(allele_order, fontsize=22, rotation=45, ha="right")
    ax.tick_params(axis="both", labelsize=18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    ax.set_axisbelow(True)
    ax.set_xlabel(group, fontsize=22, labelpad=12)

axes[0].set_ylabel("Peaks with ≥1 G4 (%)", fontsize=22)
axes[0].set_ylim(0, max(df["occ_peak_pct"].max(), df["occ_ctrl_pct"].max()) * 1.45)

handles = [
    mpatches.Patch(facecolor=PEAK_COLOR, edgecolor="black", linewidth=1.5, label="PRDM9 peaks"),
    mpatches.Patch(facecolor=CTRL_COLOR, edgecolor="black", linewidth=1.5, label="Matched control"),
]
axes[-1].legend(handles=handles, fontsize=18, frameon=True, fancybox=True,
                bbox_to_anchor=(1.0, 1.0), loc="upper left")

fig.savefig(f"{target_fig}/prmd9_g4_peaks_pct_vs_control.pdf",
            dpi=600, transparent=True, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
for col in ["pval_mcnemar", "pval_wx", "pval_perm"]:
    adj_col   = col.replace("pval", "adj_pval")
    stars_col = col.replace("pval", "stars")
    _, results_df[adj_col], _, _ = multipletests(results_df[col], method="fdr_bh")
    results_df[stars_col] = results_df[adj_col].map(starme)
results_df.to_csv(target_data / "prmd9_hotspot_data.tsv.gz", compression="gzip", sep="\t", index=False)
 
for col in ["pval_mcnemar", "pval_wx", "pval_perm"]:
    adj_col   = col.replace("pval", "adj_pval")
    stars_col = col.replace("pval", "stars")
    _, results_merged_df[adj_col], _, _ = multipletests(results_merged_df[col], method="fdr_bh")
    results_merged_df[stars_col] = results_merged_df[adj_col].map(starme)

results_merged_df.to_csv(target_data / "prmd9_hotspot_data_merged.tsv.gz", compression="gzip", sep="\t", index=False)
results_merged_df

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

UNIFIED_COLOR = "#2166ac"
GC_COLOR      = "#c94040"
NS_COLOR      = "#d3d5db"
BAR_W         = 0.28
GAP           = 0.04
VIEW_CLIP     = 2.0     # clip the background tails to this, but never hide an observation

def pval_to_stars(p):
    if p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

results_df["stars"] = results_df["adj_pval_wx"].apply(pval_to_stars)
lookup_results = {(r["allele"], r["group"]): r for r in results_df.to_dict("records")}

df_pd = results_hotspots_df.to_pandas()
allele_order = list(dict.fromkeys(
    df_pd[df_pd["group"] == "G4Hunter"]
    .sort_values("fold_enrichment", ascending=False)["allele"].tolist()
))
lookup   = {(r["allele"], r["group"]): r for r in df_pd.to_dict("records")}
datasets = ["G4Hunter", "Quadparser", "eG4"]

tab10      = plt.cm.tab10.colors
allele_clr = {a: tab10[i % 10] for i, a in enumerate(allele_order)}

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.patch.set_facecolor("white")
fig.subplots_adjust(wspace=0.15)

for idx, (ax, ds) in enumerate(zip(axes, datasets)):
    sub    = df_pd[df_pd["group"] == ds].set_index("allele").reindex(allele_order).reset_index()
    x      = np.arange(len(allele_order))
    fe     = sub["fold_enrichment"].values

    re       = np.array([lookup_results.get((a, ds), {}).get("rel_fe", np.nan) for a in allele_order])
    stars_re = [lookup_results.get((a, ds), {}).get("stars", "ns") for a in allele_order]

    stars_    = [lookup.get((a, ds), {}).get("stars", "ns") for a in allele_order]
    fe_colors = [UNIFIED_COLOR if s != "ns" else NS_COLOR for s in stars_]

    offset   = BAR_W / 2 + GAP / 2
    bars_fe  = ax.bar(x - offset, fe, width=BAR_W, color=fe_colors,
                      edgecolor="black", linewidth=1.5, alpha=0.85, zorder=3)
    bars_re  = ax.bar(x + offset, re, width=BAR_W, color=GC_COLOR,
                      edgecolor="black", linewidth=1.5, alpha=0.85, zorder=3)
    for bar in bars_re:
        bar.set_linestyle("--")

    ax.axhline(1.0, ls="--", color="gray", lw=3.0, zorder=0)

    all_vals = np.concatenate([fe[~np.isnan(fe)], re[~np.isnan(re)]])
    ymax     = all_vals.max() if len(all_vals) else 2.0

    for xi, (yi, star) in enumerate(zip(fe, stars_)):
        if not np.isnan(yi) and star != "ns":
            ax.text(xi - offset + 0.1, yi + ymax * 0.04, star,
                    ha="center", va="bottom", fontsize=14,
                    fontweight="bold", rotation=90)

    for xi, (yi, star) in enumerate(zip(re, stars_re)):
        if not np.isnan(yi) and star != "ns":
            ax.text(xi + offset + 0.1, yi + ymax * 0.04, star,
                    ha="center", va="bottom", fontsize=14,
                    fontweight="bold", rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(allele_order, rotation=45, ha="right", fontsize=20)
    ax.set_ylabel("Fold Enrichment" if idx == 0 else "", fontsize=22)
    ax.tick_params(axis="y", labelsize=18)
    ax.set_ylim(bottom=0, top=ymax * 1.75)
    ax.grid(axis="y", lw=0.5, alpha=0.4)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title(
        ds, fontsize=20, fontweight="bold", color="black", pad=14,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="whitesmoke",
                  edgecolor="black", linewidth=1.3)
    )

    # ── residual inset ────────────────────────────────────────────────────
    ax_ins = ax.inset_axes([0.15, 0.72, 0.95, 0.2])
    bg     = residuals[ds]

    obs_vals = [(lookup[(a, ds)]["residuals"], a) for a in allele_order
                if lookup.get((a, ds)) is not None
                and not np.isnan(lookup[(a, ds)]["residuals"])]
    obs_arr = np.array([v for v, _ in obs_vals]) if obs_vals else np.array([0.0])

    # view range: clip the long background tails, but always keep observations on-panel
    view_lo = min(max(bg.min() * 1.15, -VIEW_CLIP), obs_arr.min() - 0.10)
    view_hi = max(min(bg.max() * 1.15,  VIEW_CLIP), obs_arr.max() + 0.10)

    kde = gaussian_kde(bg, bw_method="scott")
    xr  = np.linspace(view_lo, view_hi, 400)
    yr  = kde(xr)

    ax_ins.fill_between(xr, yr, color="silver", alpha=0.45)
    ax_ins.plot(xr, yr, color="dimgray", lw=1.5)
    ax_ins.axvline(0, ls="--", color="gray", lw=0.9)

    obs_sorted = sorted(obs_vals, key=lambda t: t[0])
    n_obs      = len(obs_sorted)
    span       = view_hi - view_lo
    label_xs   = np.linspace(view_lo + 0.10 * span, view_hi - 0.10 * span, max(n_obs, 1))
    y_bands    = [yr.max() * 1.35, yr.max() * 2.0]

    ax_ins.set_xlim(view_lo, view_hi)
    ax_ins.set_ylim(0, yr.max() * 2.7)

    for i, (obs, allele) in enumerate(obs_sorted):
        col   = allele_clr[allele]
        y_pos = kde(obs)[0] + 0.10 * yr.max()     # relative offset, scale-free
        ax_ins.scatter([obs], [y_pos], color=col, s=65, zorder=6,
                       edgecolors="black", linewidths=0.5)
        ax_ins.annotate(
            allele, xy=(obs, y_pos), xytext=(label_xs[i], y_bands[i % 2]),
            arrowprops=dict(arrowstyle="-", color=col, lw=0.9,
                            connectionstyle="arc3,rad=0.15"),
            fontsize=14.0, color=col, fontweight="bold", ha="center", va="bottom",
        )

    ax_ins.set_xlabel("Observed − GC-predicted", fontsize=12.5)
    ax_ins.set_ylabel("Density", fontsize=12.5)
    ax_ins.tick_params(labelsize=12.5)
    ax_ins.spines["top"].set_visible(False)
    ax_ins.spines["right"].set_visible(False)
    ax_ins.set_title("Residuals", fontsize=14, pad=4)

fig.legend(
    handles=[
        mpatches.Patch(facecolor=UNIFIED_COLOR, edgecolor="black", linewidth=1.2,
                       label="Genome-wide enrichment"),
        mpatches.Patch(facecolor=GC_COLOR, edgecolor="black", linewidth=1.2,
                       linestyle="--", label="GC-adjusted enrichment"),
    ],
    fontsize=18, loc="lower center", ncol=3, frameon=True, fancybox=True,
    bbox_to_anchor=(0.5, -0.15),
)

fig.savefig(f"{target_fig}/prmd9_g4_enrichment_per_dataset.png",
            pad_inches=0.2, 
            dpi=600, 
            transparent=True, 
            bbox_inches="tight")
fig.savefig(f"{target_fig}/prmd9_g4_enrichment_per_dataset.pdf",
            pad_inches=0.2, 
            dpi=600, 
            transparent=True, 
            bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

UNIFIED_COLOR = "#2166ac"
GC_COLOR      = "#c94040"
NS_COLOR      = "#d3d5db"
BAR_W         = 0.28
GAP           = 0.04
VIEW_CLIP     = 2.0     # clip the background tails to this, but never hide an observation

def pval_to_stars(p):
    if p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

results_merged_df["stars"] = results_merged_df["adj_pval_wx"].apply(pval_to_stars)
lookup_results = {(r["allele"], r["group"]): r for r in results_merged_df.to_dict("records")}

df_pd = results_hotspots_df.to_pandas()
allele_order = list(dict.fromkeys(
    df_pd[df_pd["group"] == "G4Hunter"]
    .sort_values("fold_enrichment", ascending=False)["allele"].tolist()
))
lookup   = {(r["allele"], r["group"]): r for r in df_pd.to_dict("records")}
datasets = ["G4Hunter", "Quadparser", "eG4"]

tab10      = plt.cm.tab10.colors
allele_clr = {a: tab10[i % 10] for i, a in enumerate(allele_order)}

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)
fig.patch.set_facecolor("white")
fig.subplots_adjust(wspace=0.15)

for idx, (ax, ds) in enumerate(zip(axes, datasets)):
    sub    = df_pd[df_pd["group"] == ds].set_index("allele").reindex(allele_order).reset_index()
    x      = np.arange(len(allele_order))
    fe     = sub["fold_enrichment"].values

    re       = np.array([lookup_results.get((a, ds), {}).get("rel_fe", np.nan) for a in allele_order])
    stars_re = [lookup_results.get((a, ds), {}).get("stars", "ns") for a in allele_order]

    stars_    = [lookup.get((a, ds), {}).get("stars", "ns") for a in allele_order]
    fe_colors = [UNIFIED_COLOR if s != "ns" else NS_COLOR for s in stars_]

    offset   = BAR_W / 2 + GAP / 2
    bars_fe  = ax.bar(x - offset, fe, width=BAR_W, color=fe_colors,
                      edgecolor="black", linewidth=1.5, alpha=0.85, zorder=3)
    bars_re  = ax.bar(x + offset, re, width=BAR_W, color=GC_COLOR,
                      edgecolor="black", linewidth=1.5, alpha=0.85, zorder=3)
    for bar in bars_re:
        bar.set_linestyle("--")

    ax.axhline(1.0, ls="--", color="gray", lw=3.0, zorder=0)

    all_vals = np.concatenate([fe[~np.isnan(fe)], re[~np.isnan(re)]])
    ymax     = all_vals.max() if len(all_vals) else 2.0

    for xi, (yi, star) in enumerate(zip(fe, stars_)):
        if not np.isnan(yi) and star != "ns":
            ax.text(xi - offset + 0.1, yi + ymax * 0.04, star,
                    ha="center", va="bottom", fontsize=14,
                    fontweight="bold", rotation=90)

    for xi, (yi, star) in enumerate(zip(re, stars_re)):
        if not np.isnan(yi) and star != "ns":
            ax.text(xi + offset + 0.1, yi + ymax * 0.04, star,
                    ha="center", va="bottom", fontsize=14,
                    fontweight="bold", rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(allele_order, rotation=45, ha="right", fontsize=20)
    ax.set_ylabel("Fold Enrichment" if idx == 0 else "", fontsize=22)
    ax.tick_params(axis="y", labelsize=18)
    ax.set_ylim(bottom=0, top=ymax * 1.75)
    ax.grid(axis="y", lw=0.5, alpha=0.4)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title(
        ds, fontsize=20, fontweight="bold", color="black", pad=14,
        bbox=dict(boxstyle="round,pad=0.45", facecolor="whitesmoke",
                  edgecolor="black", linewidth=1.3)
    )

    # ── residual inset ────────────────────────────────────────────────────
    ax_ins = ax.inset_axes([0.15, 0.72, 0.95, 0.2])
    bg     = residuals[ds]

    obs_vals = [(lookup[(a, ds)]["residuals"], a) for a in allele_order
                if lookup.get((a, ds)) is not None
                and not np.isnan(lookup[(a, ds)]["residuals"])]
    obs_arr = np.array([v for v, _ in obs_vals]) if obs_vals else np.array([0.0])

    # view range: clip the long background tails, but always keep observations on-panel
    view_lo = min(max(bg.min() * 1.15, -VIEW_CLIP), obs_arr.min() - 0.10)
    view_hi = max(min(bg.max() * 1.15,  VIEW_CLIP), obs_arr.max() + 0.10)

    kde = gaussian_kde(bg, bw_method="scott")
    xr  = np.linspace(view_lo, view_hi, 400)
    yr  = kde(xr)

    ax_ins.fill_between(xr, yr, color="silver", alpha=0.45)
    ax_ins.plot(xr, yr, color="dimgray", lw=1.5)
    ax_ins.axvline(0, ls="--", color="gray", lw=0.9)

    obs_sorted = sorted(obs_vals, key=lambda t: t[0])
    n_obs      = len(obs_sorted)
    span       = view_hi - view_lo
    label_xs   = np.linspace(view_lo + 0.10 * span, view_hi - 0.10 * span, max(n_obs, 1))
    y_bands    = [yr.max() * 1.35, yr.max() * 2.0]

    ax_ins.set_xlim(view_lo, view_hi)
    ax_ins.set_ylim(0, yr.max() * 2.7)

    for i, (obs, allele) in enumerate(obs_sorted):
        col   = allele_clr[allele]
        y_pos = kde(obs)[0] + 0.10 * yr.max()     # relative offset, scale-free
        ax_ins.scatter([obs], [y_pos], color=col, s=65, zorder=6,
                       edgecolors="black", linewidths=0.5)
        ax_ins.annotate(
            allele, xy=(obs, y_pos), xytext=(label_xs[i], y_bands[i % 2]),
            arrowprops=dict(arrowstyle="-", color=col, lw=0.9,
                            connectionstyle="arc3,rad=0.15"),
            fontsize=14.0, color=col, fontweight="bold", ha="center", va="bottom",
        )

    ax_ins.set_xlabel("Observed − GC-predicted", fontsize=12.5)
    ax_ins.set_ylabel("Density", fontsize=12.5)
    ax_ins.tick_params(labelsize=12.5)
    ax_ins.spines["top"].set_visible(False)
    ax_ins.spines["right"].set_visible(False)
    ax_ins.set_title("Residuals", fontsize=14, pad=4)

fig.legend(
    handles=[
        mpatches.Patch(facecolor=UNIFIED_COLOR, edgecolor="black", linewidth=1.2,
                       label="Genome-wide enrichment"),
        mpatches.Patch(facecolor=GC_COLOR, edgecolor="black", linewidth=1.2,
                       linestyle="--", label="GC-adjusted enrichment"),
    ],
    fontsize=18, loc="lower center", ncol=3, frameon=True, fancybox=True,
    bbox_to_anchor=(0.5, -0.15),
)

fig.savefig(f"{target_fig}/prmd9_g4_enrichment_per_dataset_merged.png",
            pad_inches=0.5, dpi=600, transparent=True, bbox_inches="tight")
fig.savefig(f"{target_fig}/prmd9_g4_enrichment_per_dataset_merged.pdf",
            pad_inches=0.5, dpi=600, transparent=True, bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
target_data

## 8. Bar plot — eG4 relative fold enrichment in PRDM9 hotspots

In [ ]:
palette   = {"eG4": "#ded714", "G4Hunter": "#2166ac", "Quadparser": "#6272d4"}
BAR_NS    = "#b0b8c8"
DATASETS  = ["G4Hunter", "Quadparser", "eG4"]

allele_order = (results_df.groupby("allele")["rel_fe"]
                .mean().sort_values(ascending=False).index.tolist())

x     = np.arange(len(allele_order))
width = 0.6
ymax_all = np.nanmax(results_df["rel_fe"].values)

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(22, 7), sharey=True)
fig.patch.set_facecolor("white")

for idx, (ds, ax) in enumerate(zip(DATASETS, axes)):
    sub    = results_df[results_df["group"] == ds].set_index("allele").reindex(allele_order)
    fe     = sub["rel_fe"].values
    stars  = sub["stars_wx"].values
    colors = [palette["G4Hunter"] for _ in range(3)]

    ax.bar(x, fe, width=width, color=colors,
           edgecolor="black", linewidth=1.5, zorder=3)

    for i, (fev, star) in enumerate(zip(fe, stars)):
        if np.isnan(fev) or star == "ns":
            continue
        ax.text(x[i], fev + ymax_all * 0.03, star,
                ha="center", va="bottom", fontsize=13, fontweight="bold", zorder=5)

    ax.axhline(1.0, ls="--", lw=2.0, color="gray", zorder=0)
    ax.set_ylim(0, ymax_all * 1.1)
    ax.set_xticks(x)
    ax.set_xticklabels(allele_order, rotation=40, ha="right", fontsize=26)
    ax.tick_params(axis="y", labelsize=24)
    ax.text(0.5, 1.04, ds,
            transform=ax.transAxes,
            fontsize=26, fontweight="bold", color="black",
            ha="center", va="bottom",
            bbox=dict(boxstyle="round,pad=0.4",
                      facecolor="#DDDDDD",
                      edgecolor="#999999",
                      linewidth=1.5))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", lw=0.4, alpha=0.6)
    if idx == 0:
        ax.set_ylabel("Fold Enrichment (Hotspot / Control)", fontsize=24)

plt.tight_layout()
fig.savefig(target_fig / "prmd9_hotspot_all_datasets_fold_enrichment.pdf",
            bbox_inches="tight", transparent=True)
plt.show()

## PRMD9 Density

In [ ]:
OUT_DIR = Path(os.getenv("SCRATCH")) / "g4_t2t_revisions_data"

def load_matched_controls(out_dir: Path = OUT_DIR) -> dict:
    return {
        name: pd.read_table(out_dir / f"matched_controls_{name}_global.tsv.gz")
        for name in ["G4Hunter", "Quadparser"]
    }

matched = load_matched_controls()
matched.keys()

In [ ]:
# Controls ?
# Controls ?
matched_df = matched["G4Hunter"]
masked_chrom_sequences_G4Hunter = dict()
masked_chrom_sequences_Quadparser = dict()
masked_chrom_sequences = dict()
masked_chrom_sequences_controls = dict()
masked_chrom_sequences_eg4 = dict()

for seqID, seq in tqdm(chrom_sequences_hs1.items(), total=25):
    temp_df = matched_df[matched_df["seqID"] == seqID]
    seq_list = list(seq)
    for _, row in temp_df.iterrows():
        start, end = row["start"], row["end"]
        seq_list[start: end] = ["*"] * (end-start)
    masked_chrom_sequences[seqID] = "".join(seq_list)
    
    # Controls
    seq_list = list(seq)
    for _, row in temp_df.iterrows():
        start, end = row["control_start"], row["control_end"]
        seq_list[start: end] = ["*"] * (end-start)
    masked_chrom_sequences_controls[seqID] = "".join(seq_list)

    # G4 all
    temp_df = g4_df[g4_df["seqID"] == seqID]
    seq_list = list(seq)
    for _, row in temp_df.iterrows():
        start, end = row["start"], row["end"]
        seq_list[start: end] = ["*"] * (end-start)
    masked_chrom_sequences_G4Hunter[seqID] = "".join(seq_list)

    # Quadparser
    temp_df = regex_df[regex_df["seqID"] == seqID]
    seq_list = list(seq)
    for _, row in temp_df.iterrows():
        start, end = row["start"], row["end"]
        seq_list[start: end] = ["*"] * (end-start)
    masked_chrom_sequences_Quadparser[seqID] = "".join(seq_list)

    # eG4
    temp_df = eg4_df[eg4_df["seqID"] == seqID]
    seq_list = list(seq)
    for _, row in temp_df.iterrows():
        start, end = row["start"], row["end"]
        seq_list[start: end] = ["*"] * (end-start)
    masked_chrom_sequences_eg4[seqID] = "".join(seq_list)

In [ ]:
hotspot_controls_merged.keys()

In [ ]:
import numpy as np
WINDOW_SIZE = 1000
PRMD9_expanded = dict()

for allele in tqdm(alleles):
    PRMD9_expanded[allele] = hotspots_hs1_merged[allele].copy()
    PRMD9_expanded[allele].loc[:, "mid"] = (PRMD9_expanded[allele]["start"] + PRMD9_expanded[allele]["end"]) // 2
    PRMD9_expanded[allele].loc[:, "chrom_size"] = PRMD9_expanded[allele]["seqID"].apply(lambda x: seq_sizes_hs1[x])
    PRMD9_expanded[allele].loc[:, "expanded_start"] = np.maximum(PRMD9_expanded[allele]["mid"] - WINDOW_SIZE, 0)
    PRMD9_expanded[allele].loc[:, "expanded_end"] = np.minimum(PRMD9_expanded[allele]["mid"] + WINDOW_SIZE + 1, PRMD9_expanded[allele]["chrom_size"])
    PRMD9_expanded[allele].loc[:, f"interval_sequence_G4"] = PRMD9_expanded[allele].apply(lambda row: masked_chrom_sequences[row["seqID"]][row["expanded_start"]:row["expanded_end"]], axis=1)
    PRMD9_expanded[allele].loc[:, f"interval_sequence_Quadparser"] = PRMD9_expanded[allele].apply(lambda row: masked_chrom_sequences_Quadparser[row["seqID"]][row["expanded_start"]:row["expanded_end"]], axis=1)
    PRMD9_expanded[allele].loc[:, f"interval_control_sequence_G4"] = PRMD9_expanded[allele].apply(lambda row: masked_chrom_sequences_controls[row["seqID"]][row["expanded_start"]:row["expanded_end"]], axis=1)
    print("DONE!")

In [ ]:
# FOR REGIONS
import numpy as np
WINDOW_SIZE = 1000
PRMD9_expanded_R = dict()
PRMD9_expanded_R_Quadparser = dict()
PRMD9_expanded_control_R = dict()
alleles = hotspot_controls.keys()

for allele in tqdm(alleles):
    # G4Hunter
    PRMD9_expanded_R[allele] = hotspot_controls[allele].copy()
    PRMD9_expanded_R[allele].loc[:, "mid"] = (PRMD9_expanded_R[allele]["peak_start"] + PRMD9_expanded_R[allele]["peak_end"]) // 2
    PRMD9_expanded_R[allele].loc[:, "chrom_size"] = PRMD9_expanded_R[allele]["peak_seqID"].apply(lambda x: seq_sizes_hs1[x])
    PRMD9_expanded_R[allele].loc[:, "expanded_start"] = np.maximum(PRMD9_expanded_R[allele]["mid"] - WINDOW_SIZE, 0)
    PRMD9_expanded_R[allele].loc[:, "expanded_end"] = np.minimum(PRMD9_expanded_R[allele]["mid"] + WINDOW_SIZE + 1, PRMD9_expanded_R[allele]["chrom_size"])
    PRMD9_expanded_R[allele].loc[:, f"interval_sequence_G4"] = PRMD9_expanded_R[allele].apply(lambda row: masked_chrom_sequences_G4Hunter[row["peak_seqID"]][row["expanded_start"]:row["expanded_end"]], axis=1)

     # Control Region
    PRMD9_expanded_control_R[allele] = hotspot_controls[allele].copy()
    PRMD9_expanded_control_R[allele].loc[:, "mid"] = (PRMD9_expanded_control_R[allele]["ctrl_start"] + PRMD9_expanded_control_R[allele]["ctrl_end"]) // 2
    PRMD9_expanded_control_R[allele].loc[:, "chrom_size"] = PRMD9_expanded_control_R[allele]["ctrl_seqID"].apply(lambda x: seq_sizes_hs1[x])
    PRMD9_expanded_control_R[allele].loc[:, "expanded_start"] = np.maximum(PRMD9_expanded_control_R[allele]["mid"] - WINDOW_SIZE, 0)
    PRMD9_expanded_control_R[allele].loc[:, "expanded_end"] = np.minimum(PRMD9_expanded_control_R[allele]["mid"] + WINDOW_SIZE + 1, PRMD9_expanded_control_R[allele]["chrom_size"])
    PRMD9_expanded_control_R[allele].loc[:, f"interval_sequence_G4"] = PRMD9_expanded_control_R[allele].apply(lambda row: masked_chrom_sequences_G4Hunter[row["ctrl_seqID"]][row["expanded_start"]:row["expanded_end"]], axis=1)

    # Quadparser
    PRMD9_expanded_R_Quadparser[allele] = hotspot_controls[allele].copy()
    PRMD9_expanded_R_Quadparser[allele].loc[:, "mid"] = (PRMD9_expanded_R_Quadparser[allele]["peak_start"] + PRMD9_expanded_R_Quadparser[allele]["peak_end"]) // 2
    PRMD9_expanded_R_Quadparser[allele].loc[:, "chrom_size"] = PRMD9_expanded_R_Quadparser[allele]["peak_seqID"].apply(lambda x: seq_sizes_hs1[x])
    PRMD9_expanded_R_Quadparser[allele].loc[:, "expanded_start"] = np.maximum(PRMD9_expanded_R_Quadparser[allele]["mid"] - WINDOW_SIZE, 0)
    PRMD9_expanded_R_Quadparser[allele].loc[:, "expanded_end"] = np.minimum(PRMD9_expanded_R_Quadparser[allele]["mid"] + WINDOW_SIZE + 1, PRMD9_expanded_R_Quadparser[allele]["chrom_size"])
    PRMD9_expanded_R_Quadparser[allele].loc[:, f"interval_sequence_G4"] = PRMD9_expanded_R_Quadparser[allele].apply(lambda row: masked_chrom_sequences_Quadparser[row["peak_seqID"]][row["expanded_start"]:row["expanded_end"]], axis=1)
    print("DONE!")


In [ ]:
# G4s
full_length = 2 * WINDOW_SIZE + 1
G4_count_array = dict()
control_count_array = dict()
G4_count_array_Quadparser = dict()
mark = "*"
for allele in alleles:
    # Motif Based Comparison
    sequences = PRMD9_expanded[allele]["interval_sequence_G4"].values
    full_mask = np.array([len(s) == full_length for s in sequences])
    sequences_full = sequences[full_mask]
    seq_matrix = np.array([list(s) for s in sequences_full], dtype="U1")
    G4_count_array[allele] = np.sum(seq_matrix == mark, axis=0)

    # Control sequences
    sequences = PRMD9_expanded[allele]["interval_control_sequence_G4"].values
    full_mask = np.array([len(s) == full_length for s in sequences])
    sequences_full = sequences[full_mask]
    seq_matrix = np.array([list(s) for s in sequences_full], dtype="U1")
    control_count_array[allele] = np.sum(seq_matrix == mark, axis=0)

    # QUADPARSER
    sequences = PRMD9_expanded[allele]["interval_sequence_Quadparser"].values
    full_mask = np.array([len(s) == full_length for s in sequences])
    sequences_full = sequences[full_mask]
    seq_matrix = np.array([list(s) for s in sequences_full], dtype="U1")
    G4_count_array_Quadparser[allele] = np.sum(seq_matrix == mark, axis=0)
    print("DONE!")

In [ ]:
# G4s
full_length = 2 * WINDOW_SIZE + 1
G4_count_array_R = dict()
G4_count_array_controls_R = dict()
G4_count_array_Quadparser = dict()

mark = "*"
for allele in alleles:
    sequences = PRMD9_expanded_R[allele]["interval_sequence_G4"].values
    full_mask = np.array([len(s) == full_length for s in sequences])
    sequences_full = sequences[full_mask]
    seq_matrix = np.array([list(s) for s in sequences_full], dtype="U1")
    G4_count_array_R[allele] = np.sum(seq_matrix == mark, axis=0)

    # Control
    sequences = PRMD9_expanded_control_R[allele]["interval_sequence_G4"].values
    full_mask = np.array([len(s) == full_length for s in sequences])
    sequences_full = sequences[full_mask]
    seq_matrix = np.array([list(s) for s in sequences_full], dtype="U1")
    G4_count_array_controls_R[allele] = np.sum(seq_matrix == mark, axis=0)

    # Quadparser
    sequences = PRMD9_expanded_R_Quadparser[allele]["interval_sequence_G4"].values
    full_mask = np.array([len(s) == full_length for s in sequences])
    sequences_full = sequences[full_mask]
    seq_matrix = np.array([list(s) for s in sequences_full], dtype="U1")
    G4_count_array_Quadparser[allele] = np.sum(seq_matrix == mark, axis=0)

In [ ]:
secured = True
if secured:
    for allele in G4_count_array_R.keys():
        # PRMD9 Region
        matrix = G4_count_array_R[allele]
        target__ = target_data.joinpath(f"PRMD9_{allele}_ALL_matrix_G4.Region.tsv")
        np.savetxt(target__, matrix, delimiter=',')

        # Control Regions
        matrix = G4_count_array_controls_R[allele]
        target__ = target_data.joinpath(f"PRMD9_{allele}_ALL_matrix_Controls.Region.tsv")
        np.savetxt(target__, matrix, delimiter=',')

        # Quadparser
        matrix = G4_count_array_Quadparser[allele]
        target__ = target_data.joinpath(f"PRMD9_{allele}_ALL_matrix_Quadparser.Region.tsv")
        np.savetxt(target__, matrix, delimiter=',')

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.multitest import multipletests
from pathlib import Path

FLANK = 150
MIN_ENRICHMENT = 0.2
WINDOW_SIZE = 1000

n_alleles = len(alleles)
ncols = 2
nrows = int(np.ceil(n_alleles / ncols))
# Each allele needs a main + sig strip row, so multiply by 2
fig, axes = plt.subplots(
    nrows=nrows * 2,
    ncols=ncols,
    figsize=(16, 24),
    gridspec_kw={"height_ratios": [10, 1] * nrows}
)
fig.subplots_adjust(wspace=0.35, hspace=0.3)
palette = {"G4Hunter": "#e617d4", 
            "Control": "steelblue", 
            "Quadparser": "pink"}

for idx, allele in enumerate(alleles):
    col = idx % ncols
    row = (idx // ncols) * 2  # main row
    
    ax     = axes[row,     col]
    ax_sig = axes[row + 1, col]

    # share x between main and sig strip
    ax_sig.sharex(ax)
    x     = np.arange(-WINDOW_SIZE, WINDOW_SIZE + 1)
    n_pos = len(x)

    center = WINDOW_SIZE
    g4   = G4_count_array_R[allele].astype(float)[center - WINDOW_SIZE:center + WINDOW_SIZE + 1]
    quadparser = G4_count_array_Quadparser[allele].astype(float)[center - WINDOW_SIZE:center + WINDOW_SIZE + 1]
    ctrl = G4_count_array_controls_R[allele].astype(float)[center - WINDOW_SIZE:center + WINDOW_SIZE + 1]

    N_g4   = int(g4.sum())
    N_quadparser = int(quadparser.sum())
    N_ctrl = int(ctrl.sum())

    p_g4   = g4   / N_g4
    p_ctrl = ctrl / N_ctrl

    g4_lo,   g4_hi   = proportion_confint(g4,   N_g4,   alpha=0.05, method="wilson")
    ctrl_lo, ctrl_hi = proportion_confint(ctrl, N_ctrl, alpha=0.05, method="wilson")
    quadparser_lo,   quadparser_hi   = proportion_confint(quadparser,   N_quadparser,   alpha=0.05, method="wilson")


    g4_mean   = p_g4.mean()
    ctrl_mean = p_ctrl.mean()
    quadparser_mean = quadparser.mean()

    g4_line   = 1e2 * p_g4   / g4_mean
    ctrl_line = 1e2 * p_ctrl / ctrl_mean
    quadparser_line = 1e2 * quadparser / quadparser_mean

    g4_lo_s   = 1e2 * g4_lo   / g4_mean
    g4_hi_s   = 1e2 * g4_hi   / g4_mean
    ctrl_lo_s = 1e2 * ctrl_lo / ctrl_mean
    ctrl_hi_s = 1e2 * ctrl_hi / ctrl_mean
    quadparser_lo_s = 1e2 * quadparser_lo / quadparser_mean
    quadparser_hi_s = 1e2 * quadparser_hi / quadparser_mean

    pvals = []
    for i in range(n_pos):
        counts = np.array([g4[i],   ctrl[i]])
        nobs   = np.array([N_g4,    N_ctrl])
        _, p   = proportions_ztest(counts, nobs, alternative="larger")
        pvals.append(p)
    pvals = np.array(pvals)

    _, qvals, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")

    relative_enrichment = (p_g4 - p_ctrl) / p_ctrl
    sig = (qvals < 0.05) & (relative_enrichment > MIN_ENRICHMENT)

    # ── main panel ─────────────────────────────────────────────────────────────
    ax.plot(x, g4_line,   lw=2.0, color=palette["G4Hunter"],   label="G4Hunter", zorder=3)
    ax.plot(x, ctrl_line, lw=2.0, color=palette["Control"], label="Control",  alpha=0.4, zorder=3)
    ax.plot(x, quadparser_line, lw=2.0, color=palette["Quadparser"], label="Quadparser", zorder=0)

    ax.fill_between(x, g4_lo_s,   g4_hi_s,   color=palette["G4Hunter"],   alpha=0.18, zorder=2)
    ax.fill_between(x, ctrl_lo_s, ctrl_hi_s, color=palette["Control"], alpha=0.18, zorder=2)
    ax.fill_between(x, quadparser_lo_s, quadparser_hi_s, color=palette["Quadparser"], alpha=0.05, zorder=2)

    for i, xi in enumerate(x):
        if sig[i]:
            ax.axvspan(xi - 0.5, xi + 0.5, color="gold", alpha=0.03, zorder=1)

    ax.axvline(0,     ls="--", lw=1.5, color="black",   zorder=4)
    ax.axhline(100.0, ls="--", lw=2.5, color="black", zorder=4)

#     ax.set_title(
#         allele, fontsize=22, pad=20,
#         bbox=dict(boxstyle="round,pad=0.4", facecolor="#EEEEEE",
#                   edgecolor="#AAAAAA", linewidth=1.5)
#     )
    ax.text(0.07, 0.92, allele,
                transform=ax.transAxes,
                fontsize=18, fontweight="bold",
                va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="#EEEEEE",
                          edgecolor="#AAAAAA", linewidth=1.5))
    ax.set_xlim(xmin=-WINDOW_SIZE, xmax=WINDOW_SIZE)
    if idx%2 == 0:
        ax.set_ylabel("G4 Density (normalized)", fontsize=17)
    else:
        ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=16)
    ax.grid(lw=0.4)
    ax.set_axisbelow(True)


    handles = [mpatches.Patch(color=palette[name], label=name) for name in palette.keys()]
    ax.legend(fontsize=14, handles=handles,fancybox=True, shadow=False, loc="upper right", framealpha=0.9)
    plt.setp(ax.get_xticklabels(), visible=False)

    # ── -log10(q) twin axis ────────────────────────────────────────────────────
    ax2 = ax.twinx()
    # ax2.plot(x, -np.log10(qvals + 1e-300), color="darkgray", lw=2.0,
    #          ls=":", alpha=0.7, label="-log10(q)")
    ax2.axhline(-np.log10(0.05), ls=":", color="gray", lw=0.8)
    ax2.set_ylabel("-log10(q)", fontsize=17, color="darkgray")
    ax2.tick_params(axis="y", labelcolor="darkgray", labelsize=16)
    ax2.set_axisbelow(False)

    # ── significance strip ─────────────────────────────────────────────────────
    for i, xi in enumerate(x):
        if sig[i]:
            ax_sig.axvspan(xi - 0.5, xi + 0.5, color="gold", alpha=0.1)

    ax_sig.axvline(0, ls="--", lw=1.5, color="red")
    ax_sig.set_yticks([])
    ax_sig.set_ylabel("FDR<0.05\n+effect", fontsize=15, rotation=0, labelpad=40, va="center")
    ax_sig.tick_params(axis="x", labelsize=16)

    # only show x label on bottom strip of each column
    if (idx // ncols) == nrows - 1:
        ax_sig.set_xlabel("Position relative to PRDM9 mid (bp)", fontsize=18)

target = Path("figures_g4_t2t")
target.mkdir(exist_ok=True)
fig.savefig(target_fig / "g4_distr_relative_to_PRMD9_grid_REGIONS.pdf", 
            bbox_inches="tight", 
            transparent=True)
plt.show()

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.multitest import multipletests
from pathlib import Path

FLANK = 150
MIN_ENRICHMENT = 0.2
WINDOW_SIZE = 1000

n_alleles = len(alleles)
ncols = 2
nrows = int(np.ceil(n_alleles / ncols))

# Each allele needs a main + sig strip row, so multiply by 2
fig, axes = plt.subplots(
    nrows=nrows * 2,
    ncols=ncols,
    figsize=(20, 22),
    gridspec_kw={"height_ratios": [10, 1] * nrows}
)
fig.subplots_adjust(wspace=0.35, hspace=0.4)
palette = {"G4Hunter": "#e617d4", 
            "Control": "steelblue", 
            "Quadparser": "#9866de"}

for idx, allele in enumerate(alleles):
    col = idx % ncols
    row = (idx // ncols) * 2  # main row
    
    ax     = axes[row,     col]
    ax_sig = axes[row + 1, col]

    # share x between main and sig strip
    ax_sig.sharex(ax)
    x     = np.arange(-WINDOW_SIZE, WINDOW_SIZE + 1)
    n_pos = len(x)

    center = WINDOW_SIZE
    g4   = G4_count_array[allele].astype(float)[center - WINDOW_SIZE:center + WINDOW_SIZE + 1]
    quadparser = G4_count_array_Quadparser[allele].astype(float)[center - WINDOW_SIZE:center + WINDOW_SIZE + 1]
    ctrl = control_count_array[allele].astype(float)[center - WINDOW_SIZE:center + WINDOW_SIZE + 1]

    N_g4   = int(g4.sum())
    N_quadparser = int(quadparser.sum())
    N_ctrl = int(ctrl.sum())

    p_g4   = g4   / N_g4
    p_ctrl = ctrl / N_ctrl

    g4_lo,   g4_hi   = proportion_confint(g4,   N_g4,   alpha=0.05, method="wilson")
    ctrl_lo, ctrl_hi = proportion_confint(ctrl, N_ctrl, alpha=0.05, method="wilson")
    quadparser_lo,   quadparser_hi   = proportion_confint(quadparser,   N_quadparser,   alpha=0.05, method="wilson")


    g4_mean   = p_g4.mean()
    ctrl_mean = p_ctrl.mean()
    quadparser_mean = quadparser.mean()

    g4_line   = 1e2 * p_g4   / g4_mean
    ctrl_line = 1e2 * p_ctrl / ctrl_mean
    quadparser_line = 1e2 * quadparser / quadparser_mean

    g4_lo_s   = 1e2 * g4_lo   / g4_mean
    g4_hi_s   = 1e2 * g4_hi   / g4_mean
    ctrl_lo_s = 1e2 * ctrl_lo / ctrl_mean
    ctrl_hi_s = 1e2 * ctrl_hi / ctrl_mean
    quadparser_lo_s = 1e2 * quadparser_lo / quadparser_mean
    quadparser_hi_s = 1e2 * quadparser_hi / quadparser_mean

    pvals = []
    for i in range(n_pos):
        counts = np.array([g4[i],   ctrl[i]])
        nobs   = np.array([N_g4,    N_ctrl])
        _, p   = proportions_ztest(counts, nobs, alternative="larger")
        pvals.append(p)
    pvals = np.array(pvals)

    _, qvals, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")

    relative_enrichment = (p_g4 - p_ctrl) / p_ctrl
    sig = (qvals < 0.05) & (relative_enrichment > MIN_ENRICHMENT)

    for i, xi in enumerate(x):
        if sig[i]:
            ax.axvspan(xi - 0.5, xi + 0.5, color="gold", alpha=0.05, zorder=0)
    # ── main panel ─────────────────────────────────────────────────────────────
    ax.plot(x, g4_line,   lw=1.5, color=palette["G4Hunter"],   label="G4Hunter", zorder=3)
    ax.plot(x, ctrl_line, lw=1.5, color=palette["Control"], label="Control",  alpha=0.4, zorder=3)
    # ax.plot(x, quadparser_line, lw=1.5, color=palette["Quadparser"], label="Quadparser", zorder=0)

    ax.fill_between(x, g4_lo_s,   g4_hi_s,   color=palette["G4Hunter"],   alpha=0.18, zorder=2)
    ax.fill_between(x, ctrl_lo_s, ctrl_hi_s, color=palette["Control"], alpha=0.18, zorder=2)
    # ax.fill_between(x, quadparser_lo_s, quadparser_hi_s, color=palette["Quadparser"], alpha=0.05, zorder=3)


    ax.axvline(0,     ls="--", lw=1.5, color="black",   zorder=4)
    ax.axhline(100.0, ls="--", lw=2.5, color="black", zorder=4)

#     ax.set_title(
#         allele, fontsize=22, pad=20,
#         bbox=dict(boxstyle="round,pad=0.4", facecolor="#EEEEEE",
#                   edgecolor="#AAAAAA", linewidth=1.5)
#     )
    ax.text(0.07, 0.92, allele,
                transform=ax.transAxes,
                fontsize=20, fontweight="bold",
                va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="#EEEEEE",
                          edgecolor="#AAAAAA", linewidth=1.5))
    ax.set_xlim(xmin=-WINDOW_SIZE, xmax=WINDOW_SIZE)
    if idx%2 == 0:
        ax.set_ylabel("G4 Density (normalized)", fontsize=19)
    else:
        ax.set_ylabel("")
    ax.tick_params(axis="y", labelsize=18)
    ax.grid(lw=0.4)
    ax.set_axisbelow(True)
    ax.set_ylim(ymin=50)


    handles = [mpatches.Patch(color=palette[name], label=name) for name in palette.keys() if name != "Quadparser"]
    ax.legend(fontsize=16, handles=handles,fancybox=True, shadow=False, loc="upper right", framealpha=0.9)
    plt.setp(ax.get_xticklabels(), visible=False)

    # ── -log10(q) twin axis ────────────────────────────────────────────────────
    ax2 = ax.twinx()
    ax2.plot(x, -np.log10(qvals + 1e-300), color="darkgray", lw=2.0,
             ls=":", alpha=0.7, label="-log10(q)")
    ax2.axhline(-np.log10(0.05), ls=":", color="gray", lw=0.8)
    ax2.set_ylabel("-log10(q)", fontsize=17, color="darkgray")
    ax2.tick_params(axis="y", labelcolor="darkgray", labelsize=18)
    ax2.set_axisbelow(False)

    # ── significance strip ─────────────────────────────────────────────────────
    for i, xi in enumerate(x):
        if sig[i]:
            ax_sig.axvspan(xi - 0.5, xi + 0.5, color="gold", alpha=0.05)

    ax_sig.axvline(0, ls="--", lw=1.5, color="red")
    ax_sig.set_yticks([])
    ax_sig.set_ylabel("FDR<0.05\n+effect", fontsize=18, rotation=0, labelpad=50, va="center")
    ax_sig.tick_params(axis="x", labelsize=18)

    # only show x label on bottom strip of each column
    if (idx // ncols) == nrows - 1:
        ax_sig.set_xlabel("Position relative to PRDM9 mid (bp)", fontsize=22)

target = Path("/work/10904/nikolchanchan/vista/figures_g4_t2t_NEW")
target.mkdir(exist_ok=True)
fig.savefig(target_fig / "g4_distr_relative_to_PRMD9_grid_Motifs.pdf", 
            dpi=600, 
            bbox_inches="tight", 
            transparent=True)
plt.show()

## H-DNA

In [ ]:
from Bio.Seq import Seq
mirror_df = pd.read_table("/work/10904/nikolchanchan/ls6/g4_t2t_revisions/notebooks/chm13v2.0_MR.processed.tsv")
mirror_df.loc[:, "arm_length"] = mirror_df["sequence_of_arm"].str.len()
mirror_df.loc[:, "at_prop"] = mirror_df["sequence_of_arm"].str.count("[at]").div(mirror_df["arm_length"])
mirror_df.loc[:, "ga_prop"] = mirror_df["sequence_of_arm"].str.count("[ga]").div(mirror_df["arm_length"])
mirror_df.loc[:, "ct_prop"] = mirror_df["sequence_of_arm"].str.count("[ct]").div(mirror_df["arm_length"])

# # #
THRESHOLD_HDNA = 0.8
mirror_df.loc[:, "is_HDNA"] = (mirror_df["at_prop"] < 0.8) & (mirror_df["spacer_length"] < 8) & ((mirror_df["ga_prop"] > THRESHOLD_HDNA) | (mirror_df["ct_prop"] > THRESHOLD_HDNA))
HDNA_df = mirror_df[mirror_df["is_HDNA"]].reset_index(drop=True)
HDNA_df.loc[:, "canonical"] = HDNA_df.apply(lambda row: str(Seq(row["sequence"]).reverse_complement()) if row["ct_prop"] > THRESHOLD_HDNA else row["sequence"], axis=1)
HDNA_bed = BedTool.from_dataframe(HDNA_df[["seqID", "start", "end"]]).sort()
HDNA_df

In [ ]:
g4_bed = BedTool.from_dataframe(g4_df[["seqID", "start", "end"]]).sort()

In [ ]:
peaks_classification_df = []
COVERAGE_FIELDS = ["total_hits", "total_bases", "compartment_bases", "coverage"]
for allele in tqdm(hotspots_hs1):

    hotspot_bed = BedTool.from_dataframe(hotspots_hs1[allele][["seqID", "start", "end"]]).sort().merge().sort()
    temp_G4_df = pd.read_table(hotspot_bed.coverage(g4_bed).fn, 
                            header=None,
                            names=["seqID", "start", "end"] + COVERAGE_FIELDS)
    temp_G4_df.loc[:, "at_least_one_G4"] = (temp_G4_df["total_hits"] > 0).astype(int)
    temp_HDNA_df = pd.read_table(hotspot_bed.coverage(HDNA_bed).fn,
                            header=None,
                            names=["seqID", "start", "end"] + COVERAGE_FIELDS)
    temp_HDNA_df.loc[:, "at_least_one_HDNA"] = (temp_HDNA_df["total_hits"] > 0).astype(int)

    temp_df = temp_G4_df.merge(
                               temp_HDNA_df[["seqID", "start", "end", "at_least_one_HDNA"]], 
                               on=["seqID", "start", "end"], 
                               how="left"
                        )
    temp_df["at_least_one_G4"] = temp_df["at_least_one_G4"].astype(int)
    temp_df["at_least_one_HDNA"] = temp_df["at_least_one_HDNA"].astype(int)

    # Classify Peaks
    temp_df.loc[:, "allele"] = allele
    temp_df.loc[:, "both_motifs"] = temp_df["at_least_one_G4"] & temp_df["at_least_one_HDNA"]
    temp_df.loc[:, "peak_type"] = temp_df.apply(lambda row: "Both" if row["both_motifs"] else ("G4_only" if row["at_least_one_G4"] else ("HDNA_only" if row["at_least_one_HDNA"] else "Neither")), axis=1)
    peaks_classification_df.append(temp_df)

peaks_classification_df = pd.concat(peaks_classification_df, ignore_index=True)
peaks_classification_df

In [ ]:
%matplotlib inline
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

summary_df = (
    peaks_classification_df
    .groupby(["allele", "peak_type"], as_index=False)
    .agg(n_peaks=("peak_type", "count"))
    .pivot(index="allele", columns="peak_type", values="n_peaks")
    .fillna(0)
    .astype(int)
    .reset_index()
)

for col in ["Both", "G4_only", "HDNA_only", "Neither"]:
    if col not in summary_df.columns:
        summary_df[col] = 0

summary_df["total"] = summary_df[["Both", "G4_only", "HDNA_only", "Neither"]].sum(axis=1)
summary_df["Both_pct"]     = 100 * summary_df["Both"]      / summary_df["total"]
summary_df["G4_only_pct"]  = 100 * summary_df["G4_only"]   / summary_df["total"]
summary_df["HDNA_only_pct"]= 100 * summary_df["HDNA_only"] / summary_df["total"]
summary_df["Neither_pct"]  = 100 * summary_df["Neither"]   / summary_df["total"]

summary_df = summary_df.rename(columns={"G4_only": "G4 Only", "HDNA_only": "HDNA Only"})
print(summary_df[["allele", "Both", "G4 Only", "HDNA Only", "Neither", "total"]])

# ── Figure ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(8, 16), gridspec_kw={'height_ratios': [0.8, 2]})
axes = axes.flatten()
COLORS = {
    "G4 Only":  "#db32a3",
    "Both":     "gold",
    "HDNA Only": "#4a90d9",
    "Neither":  "lightgray"
}
ORDER = ["G4 Only", "Both", "HDNA Only", "Neither"]
bottom = np.zeros(len(summary_df))
for cat in ORDER:
    axes[0].bar(
        summary_df["allele"],
        summary_df[cat],
        bottom=bottom,
        color=COLORS[cat],
        label=cat,
        edgecolor="white",
        linewidth=0.5
    )
    bottom += summary_df[cat].values

axes[0].set_title("", fontsize=18)
axes[0].set_ylabel("Number of peaks", fontsize=20)
axes[0].set_xlabel("", fontsize=14)
axes[0].tick_params(axis="x", rotation=45, labelsize=18)
axes[0].tick_params(axis="y", labelsize=15)
axes[0].legend(fontsize=15, framealpha=0.9, bbox_to_anchor=(0.80, 1.0))
axes[0].grid(axis="y", lw=0.4)
axes[0].set_axisbelow(True)

# # ── 2. Stacked bar percentages ─────────────────────────────────────────────────
# bottom = np.zeros(len(summary_df))
# for cat in ORDER:
#     axes[1].bar(
#         summary_df["allele"],
#         summary_df[f"{cat}_pct"],
#         bottom=bottom,
#         color=COLORS[cat],
#         label=cat,
#         edgecolor="white",
#         linewidth=0.5
#     )
#     bottom += summary_df[f"{cat}_pct"].values

# axes[1].set_title("PRDM9 peak composition\n(percentage)", fontsize=18)
# axes[1].set_ylabel("% of peaks", fontsize=18)
# axes[1].set_xlabel("", fontsize=14)
# axes[1].tick_params(axis="x", rotation=45, labelsize=18)
# axes[1].tick_params(axis="y", labelsize=15)
# axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
# axes[1].legend(fontsize=12, framealpha=0.9)
# axes[1].grid(axis="y", lw=0.4)
# axes[1].set_axisbelow(True)

# ── 3. Heatmap of % G4_only, Both, HDNA_only per allele ───────────────────────
heatmap_data = summary_df.set_index("allele")[["G4_only_pct", 
                                               "Both_pct", 
                                               "HDNA_only_pct"]].rename(
    columns={"G4_only_pct": "G4 Only",
             "Both_pct":    "Both",
             "HDNA_only_pct": "H-DNA Only"}
)
im = axes[1].imshow(heatmap_data.values, 
                    aspect="auto", 
                    cmap="YlOrRd", vmin=0)
axes[1].set_xticks(range(len(heatmap_data.columns)))
axes[1].set_xticklabels(heatmap_data.columns, fontsize=22)
axes[1].set_yticks(range(len(heatmap_data.index)))
axes[1].set_yticklabels(heatmap_data.index, fontsize=20)
axes[1].set_title("Overlap % per allele", fontsize=18)
cbar = plt.colorbar(im, ax=axes[1], label="% of peaks")
cbar.ax.tick_params(labelsize=18) 
cbar.ax.set_ylabel("% of peaks", fontsize=22)
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        val = heatmap_data.values[i, j]
        axes[1].text(j, i, f"{val:.1f}%",
                     ha="center", 
                     va="center", 
                     fontsize=16,
                     color="black" if val < 50 else "white")
# Add border lines
for i in range(len(heatmap_data.index) + 1):
    axes[1].axhline(i - 0.5, color="white", linewidth=1.5)
for j in range(len(heatmap_data.columns) + 1):
    axes[1].axvline(j - 0.5, color="white", linewidth=1.5)
plt.tight_layout()
fig.savefig(target_fig / "prdm9_peak_composition_g4_hdna.pdf", 
            dpi=600,
            bbox_inches="tight", 
            transparent=True)
plt.show();